# Credit Risk Scoring Engine — Exploratory Data Analysis

## Overview & Purpose
The purpose of this notebook is to conduct a thorough Exploratory Data Analysis (EDA) on the credit risk dataset before building and training our machine learning models.

### Key Objectives:
- **Baseline Dataset:** We will initially utilize the standard **German Credit Dataset** (from the UCI Machine Learning Repository) as the empirical benchmark for retail credit default behavior.
- **Pre-Modeling Understanding:** Develop a granular understanding of financial and demographic credit attributes (loan duration, credit history, account status, savings, age, employment duration) prior to feature engineering and pipeline construction.
- **Planned Analysis Workflow:**
  1. **Data Loading & Verification:** Ingest the raw credit dataset and assign standardized feature headers.
  2. **Structure Inspection:** Check sample records, dimensions, feature data types, and summary statistics.
  3. **Data Quality Checks:** Verify missing value counts, unexpected categorical values, and duplicate records.
  4. **Target Distribution Analysis:** Measure the class imbalance between good credit (non-defaults) and bad credit (defaults).
  5. **Exploratory Visualizations:** Examine risk distributions, numerical correlation matrices, and categorical default patterns.

## 1. Load the Dataset

In this section, we load the raw German Credit Dataset from `data/raw/german_credit.csv` into a Pandas `DataFrame` (`df`).

A **DataFrame** is a 2-dimensional tabular data structure with labeled rows and columns (analogous to a spreadsheet or SQL table) that enables structured inspection, slicing, and statistical analysis of credit risk features.

In [3]:
import pandas as pd

# Load the raw dataset into a Pandas DataFrame
df = pd.read_csv("../data/raw/german_credit.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

### 1.1 First 5 Rows
We inspect the top 5 records to verify attribute structure and values.

In [5]:
df.head()

### 1.2 Last 5 Rows
We inspect the bottom 5 records to ensure the full range of 1,000 observations is intact.

In [7]:
df.tail()

### 1.3 Column Names
Listing all 21 column headers present in the DataFrame.

In [9]:
df.columns

## 2. Understanding the Dataset Structure

Before building an ML model, we need to understand the data types, dimensions, and data quality. Inspecting data types (`dtypes`) and summary information ensures that all columns are parsed correctly and that numerical columns are not mistakenly treated as text strings.

In [11]:
print("Dataset Shape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nDataset Information:")
df.info()

print("\nMemory Usage:")
print(df.memory_usage(deep=True).sum(), "bytes")

### Numerical and Categorical Features

* **Numerical features:** Contain measurable numbers (e.g., loan duration, credit amount, age, installment rate).
* **Categorical features:** Contain categories/codes representing qualitative attributes (e.g., checking status, loan purpose, employment duration).
* **Encoding Requirement:** Machine learning models usually require categorical data to be encoded into numeric formats (such as one-hot encoding) before training.

In [13]:
numerical_columns = df.select_dtypes(include="number").columns
categorical_columns = df.select_dtypes(exclude="number").columns

print("Numerical columns:")
print(list(numerical_columns))

print("\nCategorical columns:")
print(list(categorical_columns))

## 3. Missing Value Analysis

Before training a machine-learning model, we need to check whether any applicant records contain missing values. Missing values can cause errors during preprocessing or affect model performance.

In [15]:
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

print("\nTotal missing values:")
print(missing_values.sum())

In [16]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

print("\nMissing percentage by column:")
print(missing_percentage)

In [17]:
if missing_values.sum() == 0:
    print("\nResult: No missing values found.")
else:
    print("\nResult: Missing values are present.")

## 4. Target Variable Analysis

The **target variable** is the outcome the machine-learning model will learn to predict. In this project, the target represents whether an applicant is classified as good or bad credit in the original German Credit Dataset.

### Raw Target Encoding:
* `1` = **Good Credit / Non-default** (Applicant paid back credit duly)
* `2` = **Bad Credit / Default** (Applicant had delays or defaulted on payments)

In [19]:
print("Target value counts:")
print(df["credit_risk"].value_counts().sort_index())

print("\nTarget percentages:")
print(
    df["credit_risk"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

In [20]:
good_count = (df["credit_risk"] == 1).sum()
bad_count = (df["credit_risk"] == 2).sum()

total = len(df)

print("\nGood credit applicants:", good_count)
print("Bad credit applicants:", bad_count)
print("Total applicants:", total)

print("\nBad credit rate:", round((bad_count / total) * 100, 2), "%")
print("Good credit rate:", round((good_count / total) * 100, 2), "%")

### Target Class Distribution Visualization
A bar chart visualizing the frequency comparison between Good Credit and Bad Credit applicants.

In [22]:
import matplotlib.pyplot as plt

labels = ["Good Credit", "Bad Credit"]
counts = [good_count, bad_count]
colors = ["#2ecc71", "#e74c3c"]

plt.figure(figsize=(7, 5))
plt.bar(labels, counts, color=colors)
plt.title("Credit Risk Class Distribution")
plt.xlabel("Credit Risk Class")
plt.ylabel("Number of Applicants")
plt.show()

## 5. Numerical Feature Analysis

Descriptive statistics help us understand the range, central tendency, and spread of numerical applicant features before preprocessing or model training. Metrics such as mean, standard deviation, median, quartiles, skewness, and histograms reveal key distributional properties like heavy right tails or extreme values.

In [24]:
numerical_features = [
    "duration_in_months",
    "credit_amount",
    "installment_rate",
    "present_residence_since",
    "age_in_years",
    "existing_credits",
    "num_people_liable"
]

df[numerical_features].describe().T

In [25]:
print("Median values:")
print(df[numerical_features].median())

In [26]:
print("\nSkewness:")
print(df[numerical_features].skew().sort_values(ascending=False))

### Numerical Feature Distributions
Histograms visualizing the empirical distribution and frequency spread for each numerical applicant feature.

In [28]:
df[numerical_features].hist(
    figsize=(14, 10),
    bins=20
)

plt.suptitle("Distributions of Numerical Features")
plt.tight_layout()
plt.show()

## 6. Categorical Feature Analysis

Categorical features represent groups or categories rather than numerical measurements. We inspect their frequency distributions before deciding how they should be encoded for machine learning.

In [30]:
categorical_features = [
    "status_checking_account",
    "credit_history",
    "purpose",
    "savings_account",
    "present_employment_since",
    "personal_status_sex",
    "other_debtors_guarantors",
    "property",
    "other_installment_plans",
    "housing",
    "job",
    "telephone",
    "foreign_worker"
]

for column in categorical_features:
    print(f"\n{'=' * 60}")
    print(f"{column}")
    print(f"Number of unique categories: {df[column].nunique()}")
    print(df[column].value_counts())

In [31]:
category_summary = pd.DataFrame({
    "unique_categories": df[categorical_features].nunique(),
    "most_common_category": df[categorical_features].mode().iloc[0],
})

category_summary

In [32]:
for column in categorical_features:
    counts = df[column].value_counts()
    rarest_category = counts.idxmin()
    rarest_count = counts.min()
    print(
        f"{column}: "
        f"rarest category = {rarest_category}, "
        f"count = {rarest_count}"
    )

## 7. Features and Target

In supervised machine learning, **features** are the input variables used by the model, while the **target** is the outcome that the model learns to predict.

In [34]:
feature_columns = [column for column in df.columns if column != "credit_risk"]

target_column = "credit_risk"

print("Number of features:", len(feature_columns))
print("\nFeature columns:")
print(feature_columns)

print("\nTarget column:")
print(target_column)

In [35]:
X = df[feature_columns]
y = df[target_column]

print("\nX shape:", X.shape)
print("y shape:", y.shape)

In [36]:
print("\nX contains target column:", "credit_risk" in X.columns)
print("Target unique values:", sorted(y.unique()))

## 8. Duplicate Row Analysis

Duplicate rows are identical records that appear more than once in a dataset. We check for duplicates before model training because duplicated records can distort data distributions and may contribute to train-test leakage if the same record appears in different splits.

In [38]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)
print(
    "Percentage of duplicate rows:",
    round((duplicate_count / len(df)) * 100, 2),
    "%"
)

In [39]:
if duplicate_count > 0:
    print("\nDuplicate rows:")
    display(df[df.duplicated(keep=False)])
else:
    print("\nNo duplicate rows found.")

## 9. Data Validity and Range Checks

A dataset can contain no missing values or duplicates and still contain invalid or unexpected values. We therefore check numerical ranges and categorical values before preprocessing.

In [41]:
numerical_features = [
    "duration_in_months",
    "credit_amount",
    "installment_rate",
    "present_residence_since",
    "age_in_years",
    "existing_credits",
    "num_people_liable"
]

range_check = pd.DataFrame({
    "min": df[numerical_features].min(),
    "max": df[numerical_features].max(),
    "unique_values": df[numerical_features].nunique()
})

print(range_check)

In [42]:
checks = {
    "age below 18": (df["age_in_years"] < 18).sum(),
    "age above 100": (df["age_in_years"] > 100).sum(),
    "loan duration <= 0": (df["duration_in_months"] <= 0).sum(),
    "credit amount <= 0": (df["credit_amount"] <= 0).sum(),
    "installment rate outside 1-4": (
        ~df["installment_rate"].between(1, 4)
    ).sum(),
    "residence outside 1-4": (
        ~df["present_residence_since"].between(1, 4)
    ).sum(),
    "existing credits outside 1-4": (
        ~df["existing_credits"].between(1, 4)
    ).sum(),
    "people liable outside 1-2": (
        ~df["num_people_liable"].between(1, 2)
    ).sum()
}

print("\nValidity checks:")
for check, count in checks.items():
    print(f"{check}: {count}")

In [43]:
print("\nUnique categorical values:")

for column in categorical_features:
    print(f"\n{column}:")
    print(sorted(df[column].unique()))

In [44]:
print("\nTarget values:")
print(sorted(df["credit_risk"].unique()))

## 10. Original UCI Feature Definitions

The table below details the authoritative definitions and encodings for all 20 predictor features and the target variable, sourced directly from `data/raw/german.doc`.

| Feature | Type | Code | Original Meaning |
| :--- | :--- | :--- | :--- |
| **`status_checking_account`** | Categorical | `A11`<br>`A12`<br>`A13`<br>`A14` | Status of existing checking account:<br>• `A11`: < 0 DM (overdrawn/negative balance)<br>• `A12`: 0 <= ... < 200 DM<br>• `A13`: >= 200 DM / salary assignments for at least 1 year<br>• `A14`: no checking account |
| **`duration_in_months`** | Numerical | — | Duration of loan repayment in months (Range: 4 to 72 months) |
| **`credit_history`** | Categorical | `A30`<br>`A31`<br>`A32`<br>`A33`<br>`A34` | Past credit repayment history:<br>• `A30`: no credits taken / all credits paid back duly<br>• `A31`: all credits at this bank paid back duly<br>• `A32`: existing credits paid back duly till now<br>• `A33`: delay in paying off in the past<br>• `A34`: critical account / other credits existing (not at this bank) |
| **`purpose`** | Categorical | `A40`<br>`A41`<br>`A42`<br>`A43`<br>`A44`<br>`A45`<br>`A46`<br>`A47`<br>`A48`<br>`A49`<br>`A410` | Loan purpose:<br>• `A40`: car (new)<br>• `A41`: car (used)<br>• `A42`: furniture/equipment<br>• `A43`: radio/television<br>• `A44`: domestic appliances<br>• `A45`: repairs<br>• `A46`: education<br>• `A47`: vacation (noted in doc as "does not exist?")<br>• `A48`: retraining<br>• `A49`: business<br>• `A410`: others |
| **`credit_amount`** | Numerical | — | Total credit principal requested in Deutsche Mark (DM) (Range: 250 to 18,424 DM) |
| **`savings_account`** | Categorical | `A61`<br>`A62`<br>`A63`<br>`A64`<br>`A65` | Savings account / bonds balance:<br>• `A61`: < 100 DM<br>• `A62`: 100 <= ... < 500 DM<br>• `A63`: 500 <= ... < 1000 DM<br>• `A64`: >= 1000 DM<br>• `A65`: unknown / no savings account |
| **`present_employment_since`** | Categorical | `A71`<br>`A72`<br>`A73`<br>`A74`<br>`A75` | Tenure with present employer:<br>• `A71`: unemployed<br>• `A72`: < 1 year<br>• `A73`: 1 <= ... < 4 years<br>• `A74`: 4 <= ... < 7 years<br>• `A75`: >= 7 years |
| **`installment_rate`** | Numerical | — | Installment rate in percentage of disposable income (Discrete tiers: 1, 2, 3, 4 %) |
| **`personal_status_sex`** | Categorical | `A91`<br>`A92`<br>`A93`<br>`A94`<br>`A95` | Marital status and sex:<br>• `A91`: male : divorced/separated<br>• `A92`: female : divorced/separated/married<br>• `A93`: male : single<br>• `A94`: male : married/widowed<br>• `A95`: female : single (documented in schema, unobserved in dataset) |
| **`other_debtors_guarantors`** | Categorical | `A101`<br>`A102`<br>`A103` | Co-debtors or guarantors:<br>• `A101`: none<br>• `A102`: co-applicant<br>• `A103`: guarantor |
| **`present_residence_since`** | Numerical | — | Duration at present residence in years / discrete bracket (Values: 1, 2, 3, 4) |
| **`property`** | Categorical | `A121`<br>`A122`<br>`A123`<br>`A124` | Most valuable asset / collateral property owned:<br>• `A121`: real estate<br>• `A122`: if not A121 : building society savings agreement / life insurance<br>• `A123`: if not A121/A122 : car or other, not in savings<br>• `A124`: unknown / no property |
| **`age_in_years`** | Numerical | — | Applicant age in years (Range: 19 to 75 years) |
| **`other_installment_plans`** | Categorical | `A141`<br>`A142`<br>`A143` | Other existing installment liabilities:<br>• `A141`: bank<br>• `A142`: stores<br>• `A143`: none |
| **`housing`** | Categorical | `A151`<br>`A152`<br>`A153` | Housing accommodation type:<br>• `A151`: rent<br>• `A152`: own home<br>• `A153`: for free |
| **`existing_credits`** | Numerical | — | Number of existing credits at this bank (Values: 1, 2, 3, 4) |
| **`job`** | Categorical | `A171`<br>`A172`<br>`A173`<br>`A174` | Employment and skill qualification type:<br>• `A171`: unemployed / unskilled - non-resident<br>• `A172`: unskilled - resident<br>• `A173`: skilled employee / official<br>• `A174`: management / self-employed / highly qualified employee / officer |
| **`num_people_liable`** | Numerical | — | Number of persons financially dependent on applicant for maintenance (Values: 1, 2) |
| **`telephone`** | Categorical | `A191`<br>`A192` | Landline/registered telephone availability:<br>• `A191`: none<br>• `A192`: yes, registered under customer's name |
| **`foreign_worker`** | Categorical | `A201`<br>`A202` | Foreign worker status:<br>• `A201`: yes<br>• `A202`: no |
| **`credit_risk`** *(Target)* | Categorical / Binary | `1`<br>`2` | Ground truth credit outcome:<br>• `1`: **Good Credit / Non-default** (700 applicants)<br>• `2`: **Bad Credit / Default** (300 applicants) |

## 11. Default Rate by Age Group

Age is a numerical applicant characteristic. Instead of assuming that age causes credit risk, we will examine whether different age groups show different historical bad-credit rates in this dataset.

In [47]:
age_bins = [18, 25, 35, 45, 55, 100]
age_labels = ["18-25", "26-35", "36-45", "46-55", "56+"]

df["age_group"] = pd.cut(
    df["age_in_years"],
    bins=age_bins,
    labels=age_labels,
    right=True
)

age_risk = (
    df.groupby("age_group", observed=False)
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

age_risk["bad_credit_rate"] = (
    age_risk["bad_credit"] /
    age_risk["applicants"] * 100
).round(2)

print(age_risk)

In [48]:
plt.figure(figsize=(8, 5))

plt.bar(
    age_risk.index.astype(str),
    age_risk["bad_credit_rate"]
)

plt.title("Bad-Credit Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Bad-Credit Rate (%)")

plt.tight_layout()
plt.show()

In [49]:
plt.figure(figsize=(8, 5))

plt.bar(
    age_risk.index.astype(str),
    age_risk["applicants"]
)

plt.title("Applicants by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

## 12. Default Rate by Loan Duration

Loan duration represents the repayment period of the credit. We will examine whether different duration groups have different historical bad-credit rates in the dataset. This is an association analysis, not a causal analysis.

In [51]:
duration_bins = [0, 12, 24, 36, 48, float("inf")]
duration_labels = [
    "4-12 months",
    "13-24 months",
    "25-36 months",
    "37-48 months",
    "49+ months"
]

df["duration_group"] = pd.cut(
    df["duration_in_months"],
    bins=duration_bins,
    labels=duration_labels,
    include_lowest=True
)

duration_risk = (
    df.groupby("duration_group", observed=False)
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

duration_risk["bad_credit_rate"] = (
    duration_risk["bad_credit"] /
    duration_risk["applicants"] * 100
).round(2)

print(duration_risk)

In [52]:
plt.figure(figsize=(9, 5))

plt.bar(
    duration_risk.index.astype(str),
    duration_risk["bad_credit_rate"]
)

plt.title("Bad-Credit Rate by Loan Duration")
plt.xlabel("Loan Duration")
plt.ylabel("Bad-Credit Rate (%)")

plt.tight_layout()
plt.show()

In [53]:
plt.figure(figsize=(9, 5))

plt.bar(
    duration_risk.index.astype(str),
    duration_risk["applicants"]
)

plt.title("Applicants by Loan Duration")
plt.xlabel("Loan Duration")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

## 13. Default Rate by Credit Amount

Credit amount is strongly right-skewed in the dataset. Therefore, quartile-based groups allow us to compare applicants across relatively equal-sized loan-amount segments.

In [55]:
df["credit_amount_group"] = pd.qcut(
    df["credit_amount"],
    q=4,
    labels=["Q1 - Lowest", "Q2", "Q3", "Q4 - Highest"]
)

credit_amount_risk = (
    df.groupby("credit_amount_group", observed=False)
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

credit_amount_risk["bad_credit_rate"] = (
    credit_amount_risk["bad_credit"] /
    credit_amount_risk["applicants"] * 100
).round(2)

print(credit_amount_risk)

In [56]:
print("\nCredit amount quartile boundaries:")
print(df["credit_amount"].quantile([0, 0.25, 0.50, 0.75, 1.00]))

In [57]:
plt.figure(figsize=(9, 5))

plt.bar(
    credit_amount_risk.index.astype(str),
    credit_amount_risk["bad_credit_rate"]
)

plt.title("Bad-Credit Rate by Credit Amount Quartile")
plt.xlabel("Credit Amount Group")
plt.ylabel("Bad-Credit Rate (%)")

plt.tight_layout()
plt.show()

In [58]:
plt.figure(figsize=(8, 5))

df.boxplot(
    column="credit_amount",
    by="credit_risk"
)

plt.title("Credit Amount Distribution by Credit Risk")
plt.suptitle("")
plt.xlabel("Credit Risk (1 = Good, 2 = Bad)")
plt.ylabel("Credit Amount (DM)")

plt.tight_layout()
plt.show()

## 14. Default Rate by Credit History

Credit history describes an applicant's previous credit repayment experience. We compare historical bad-credit rates across the different credit-history categories to identify whether past repayment behavior is associated with current credit risk.

In [60]:
credit_history_risk = (
    df.groupby("credit_history")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

credit_history_risk["bad_credit_rate"] = (
    credit_history_risk["bad_credit"] /
    credit_history_risk["applicants"] * 100
).round(2)

print(credit_history_risk)

In [61]:
overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

print("\nOverall bad-credit rate:", round(overall_bad_rate, 2), "%")

credit_history_risk["difference_from_overall"] = (
    credit_history_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print("\nCredit history risk relative to portfolio:")
print(credit_history_risk)

In [62]:
plt.figure(figsize=(10, 5))

plt.bar(
    credit_history_risk.index.astype(str),
    credit_history_risk["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Bad-Credit Rate by Credit History")
plt.xlabel("Credit History Code")
plt.ylabel("Bad-Credit Rate (%)")
plt.legend()

plt.tight_layout()
plt.show()

## 15. Default Rate by Checking Account Status

Checking-account status provides information about an applicant's existing account position in the original dataset. We compare historical bad-credit rates across these categories to identify associations with credit risk.

In [64]:
checking_risk = (
    df.groupby("status_checking_account")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

checking_risk["bad_credit_rate"] = (
    checking_risk["bad_credit"] /
    checking_risk["applicants"] * 100
).round(2)

print(checking_risk)

In [65]:
overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

checking_risk["difference_from_overall"] = (
    checking_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print("\nRisk relative to overall portfolio:")
print(checking_risk)

In [66]:
plt.figure(figsize=(9, 5))

plt.bar(
    checking_risk.index.astype(str),
    checking_risk["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Bad-Credit Rate by Checking Account Status")
plt.xlabel("Checking Account Status")
plt.ylabel("Bad-Credit Rate (%)")
plt.legend()

plt.tight_layout()
plt.show()

In [67]:
plt.figure(figsize=(9, 5))

plt.bar(
    checking_risk.index.astype(str),
    checking_risk["applicants"]
)

plt.title("Applicants by Checking Account Status")
plt.xlabel("Checking Account Status")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

## 16. Default Rate by Savings Account Status

Savings account status provides information about the applicant's recorded savings or bonds balance in the original dataset. We compare historical bad-credit rates across savings categories to identify associations with credit risk.

In [69]:
savings_risk = (
    df.groupby("savings_account")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

savings_risk["bad_credit_rate"] = (
    savings_risk["bad_credit"] /
    savings_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

savings_risk["difference_from_overall"] = (
    savings_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print(savings_risk)

In [70]:
plt.figure(figsize=(9, 5))

plt.bar(
    savings_risk.index.astype(str),
    savings_risk["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Bad-Credit Rate by Savings Account Status")
plt.xlabel("Savings Account Status")
plt.ylabel("Bad-Credit Rate (%)")
plt.legend()

plt.tight_layout()
plt.show()

In [71]:
plt.figure(figsize=(9, 5))

plt.bar(
    savings_risk.index.astype(str),
    savings_risk["applicants"]
)

plt.title("Applicants by Savings Account Status")
plt.xlabel("Savings Account Status")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

## 17. Default Rate by Employment Duration

Employment duration provides information about the applicant's employment stability in the original dataset. We compare historical bad-credit rates across employment-duration categories to identify associations with credit risk.

In [73]:
employment_risk = (
    df.groupby("present_employment_since")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

employment_risk["bad_credit_rate"] = (
    employment_risk["bad_credit"] /
    employment_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

employment_risk["difference_from_overall"] = (
    employment_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print(employment_risk)

In [74]:
plt.figure(figsize=(9, 5))

plt.bar(
    employment_risk.index.astype(str),
    employment_risk["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Bad-Credit Rate by Employment Duration")
plt.xlabel("Employment Duration Category")
plt.ylabel("Bad-Credit Rate (%)")
plt.legend()

plt.tight_layout()
plt.show()

In [75]:
plt.figure(figsize=(9, 5))

plt.bar(
    employment_risk.index.astype(str),
    employment_risk["applicants"]
)

plt.title("Applicants by Employment Duration")
plt.xlabel("Employment Duration Category")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

## 18. Default Rate by Loan Purpose

Loan purpose describes the reason for which the applicant requested credit. We compare historical bad-credit rates across purposes to identify potential associations between loan purpose and credit risk.

In [77]:
purpose_risk = (
    df.groupby("purpose")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

purpose_risk["bad_credit_rate"] = (
    purpose_risk["bad_credit"] /
    purpose_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

purpose_risk["difference_from_overall"] = (
    purpose_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print(purpose_risk)

In [78]:
purpose_names = {
    "A40": "New car",
    "A41": "Used car",
    "A42": "Furniture/equipment",
    "A43": "Radio/TV",
    "A44": "Domestic appliances",
    "A45": "Repairs",
    "A46": "Education",
    "A48": "Retraining",
    "A49": "Business",
    "A410": "Other"
}

purpose_display = purpose_risk.copy()

purpose_display["purpose_name"] = (
    purpose_display.index.map(purpose_names)
)

print(
    purpose_display[
        [
            "purpose_name",
            "applicants",
            "bad_credit",
            "bad_credit_rate",
            "difference_from_overall"
        ]
    ]
)

In [79]:
plt.figure(figsize=(12, 6))

plt.bar(
    purpose_display["purpose_name"],
    purpose_display["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Bad-Credit Rate by Loan Purpose")
plt.xlabel("Loan Purpose")
plt.ylabel("Bad-Credit Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.legend()

plt.tight_layout()
plt.show()

In [80]:
plt.figure(figsize=(12, 6))

plt.bar(
    purpose_display["purpose_name"],
    purpose_display["applicants"]
)

plt.title("Applicants by Loan Purpose")
plt.xlabel("Loan Purpose")
plt.ylabel("Number of Applicants")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 19. Default Rate by Housing Status

Housing status describes the applicant's accommodation situation in the original dataset. We compare historical bad-credit rates across housing categories to identify potential associations with credit risk.

In [82]:
housing_risk = (
    df.groupby("housing")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

housing_risk["bad_credit_rate"] = (
    housing_risk["bad_credit"] /
    housing_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

housing_risk["difference_from_overall"] = (
    housing_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print(housing_risk)

In [83]:
housing_names = {
    "A151": "Rent",
    "A152": "Own home",
    "A153": "For free"
}

housing_display = housing_risk.copy()

housing_display["housing_name"] = (
    housing_display.index.map(housing_names)
)

print(
    housing_display[
        [
            "housing_name",
            "applicants",
            "bad_credit",
            "bad_credit_rate",
            "difference_from_overall"
        ]
    ]
)

In [84]:
plt.figure(figsize=(8, 5))

plt.bar(
    housing_display["housing_name"],
    housing_display["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Bad-Credit Rate by Housing Status")
plt.xlabel("Housing Status")
plt.ylabel("Bad-Credit Rate (%)")
plt.legend()

plt.tight_layout()
plt.show()

In [85]:
plt.figure(figsize=(8, 5))

plt.bar(
    housing_display["housing_name"],
    housing_display["applicants"]
)

plt.title("Applicants by Housing Status")
plt.xlabel("Housing Status")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

## 20. Default Rate by Existing Credits

Existing credits represents the number of credits the applicant already has at the bank. We compare historical bad-credit rates across the observed credit counts to identify whether existing credit exposure is associated with current credit risk.

In [87]:
existing_credit_risk = (
    df.groupby("existing_credits")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

existing_credit_risk["bad_credit_rate"] = (
    existing_credit_risk["bad_credit"] /
    existing_credit_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

existing_credit_risk["difference_from_overall"] = (
    existing_credit_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print(existing_credit_risk)

In [88]:
plt.figure(figsize=(8, 5))

plt.bar(
    existing_credit_risk.index.astype(str),
    existing_credit_risk["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Bad-Credit Rate by Number of Existing Credits")
plt.xlabel("Number of Existing Credits")
plt.ylabel("Bad-Credit Rate (%)")
plt.legend()

plt.tight_layout()
plt.show()

In [89]:
plt.figure(figsize=(8, 5))

plt.bar(
    existing_credit_risk.index.astype(str),
    existing_credit_risk["applicants"]
)

plt.title("Applicants by Number of Existing Credits")
plt.xlabel("Number of Existing Credits")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

## 21. Default Rate by Personal Status and Sex

The original dataset combines sex and personal/marital status into one categorical variable. We analyze the historical bad-credit rates across the observed categories to understand the dataset and identify potential fairness considerations.

In [91]:
personal_status_risk = (
    df.groupby("personal_status_sex")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

personal_status_risk["bad_credit_rate"] = (
    personal_status_risk["bad_credit"] /
    personal_status_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

personal_status_risk["difference_from_overall"] = (
    personal_status_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print(personal_status_risk)

In [92]:
personal_status_names = {
    "A91": "Male - Divorced/Separated",
    "A92": "Female - Divorced/Separated/Married",
    "A93": "Male - Single",
    "A94": "Male - Married/Widowed"
}

personal_status_display = personal_status_risk.copy()

personal_status_display["category_name"] = (
    personal_status_display.index.map(personal_status_names)
)

print(
    personal_status_display[
        [
            "category_name",
            "applicants",
            "bad_credit",
            "bad_credit_rate",
            "difference_from_overall"
        ]
    ]
)

In [93]:
plt.figure(figsize=(11, 5))

plt.bar(
    personal_status_display["category_name"],
    personal_status_display["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Historical Bad-Credit Rate by Personal Status/Sex")
plt.xlabel("Personal Status / Sex Category")
plt.ylabel("Bad-Credit Rate (%)")
plt.xticks(rotation=25, ha="right")
plt.legend()

plt.tight_layout()
plt.show()

In [94]:
plt.figure(figsize=(11, 5))

plt.bar(
    personal_status_display["category_name"],
    personal_status_display["applicants"]
)

plt.title("Applicants by Personal Status/Sex Category")
plt.xlabel("Personal Status / Sex Category")
plt.ylabel("Number of Applicants")
plt.xticks(rotation=25, ha="right")

plt.tight_layout()
plt.show()

## 22. Default Rate by Property Ownership

Property represents the most valuable property or collateral category recorded for the applicant. We compare historical bad-credit rates across property categories to understand whether property ownership is associated with credit risk.

In [96]:
property_risk = (
    df.groupby("property")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

property_risk["bad_credit_rate"] = (
    property_risk["bad_credit"] /
    property_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

property_risk["difference_from_overall"] = (
    property_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

property_names = {
    "A121": "Real Estate",
    "A122": "Building Society / Life Insurance",
    "A123": "Car or Other",
    "A124": "Unknown / No Property"
}

property_display = property_risk.copy()

property_display["category_name"] = (
    property_display.index.map(property_names)
)

print(
    property_display[
        [
            "category_name",
            "applicants",
            "bad_credit",
            "bad_credit_rate",
            "difference_from_overall"
        ]
    ]
)

In [97]:
plt.figure(figsize=(10, 5))

plt.bar(
    property_display["category_name"],
    property_display["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Historical Bad-Credit Rate by Property Category")
plt.xlabel("Property Category")
plt.ylabel("Bad-Credit Rate (%)")
plt.xticks(rotation=20, ha="right")
plt.legend()

plt.tight_layout()
plt.show()

In [98]:
plt.figure(figsize=(10, 5))

plt.bar(
    property_display["category_name"],
    property_display["applicants"]
)

plt.title("Applicants by Property Category")
plt.xlabel("Property Category")
plt.ylabel("Number of Applicants")
plt.xticks(rotation=20, ha="right")

plt.tight_layout()
plt.show()

## 23. Default Rate by Job / Employment Qualification

Job represents the applicant's employment qualification category in the original UCI dataset. We compare historical bad-credit rates across job categories to understand whether employment qualification is associated with credit risk.

In [100]:
job_risk = (
    df.groupby("job")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

job_risk["bad_credit_rate"] = (
    job_risk["bad_credit"] /
    job_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

job_risk["difference_from_overall"] = (
    job_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

job_names = {
    "A171": "Unemployed / Unskilled Non-resident",
    "A172": "Unskilled Resident",
    "A173": "Skilled Employee / Official",
    "A174": "Management / Self-employed / Highly Qualified"
}

job_display = job_risk.copy()

job_display["category_name"] = (
    job_display.index.map(job_names)
)

print(
    job_display[
        [
            "category_name",
            "applicants",
            "bad_credit",
            "bad_credit_rate",
            "difference_from_overall"
        ]
    ]
)

In [101]:
plt.figure(figsize=(12, 5))

plt.bar(
    job_display["category_name"],
    job_display["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Historical Bad-Credit Rate by Job Category")
plt.xlabel("Job / Employment Qualification")
plt.ylabel("Bad-Credit Rate (%)")
plt.xticks(rotation=20, ha="right")
plt.legend()

plt.tight_layout()
plt.show()

In [102]:
plt.figure(figsize=(12, 5))

plt.bar(
    job_display["category_name"],
    job_display["applicants"]
)

plt.title("Applicants by Job Category")
plt.xlabel("Job / Employment Qualification")
plt.ylabel("Number of Applicants")
plt.xticks(rotation=20, ha="right")

plt.tight_layout()
plt.show()

## 24. Default Rate by Number of Dependents

Number of dependents represents the number of people financially dependent on the applicant. We compare historical bad-credit rates between the observed groups to investigate whether the number of dependents is associated with credit risk.

In [104]:
dependent_risk = (
    df.groupby("num_people_liable")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

dependent_risk["bad_credit_rate"] = (
    dependent_risk["bad_credit"] /
    dependent_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

dependent_risk["difference_from_overall"] = (
    dependent_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print(dependent_risk)

In [105]:
plt.figure(figsize=(8, 5))

plt.bar(
    dependent_risk.index.astype(str),
    dependent_risk["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Historical Bad-Credit Rate by Number of Dependents")
plt.xlabel("Number of Dependents")
plt.ylabel("Bad-Credit Rate (%)")
plt.legend()

plt.tight_layout()
plt.show()

In [106]:
plt.figure(figsize=(8, 5))

plt.bar(
    dependent_risk.index.astype(str),
    dependent_risk["applicants"]
)

plt.title("Applicants by Number of Dependents")
plt.xlabel("Number of Dependents")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

## 25. Default Rate by Installment Rate

Installment rate represents an ordinal bracket describing the loan installment burden relative to the applicant's disposable income. We compare historical bad-credit rates across the four observed tiers.

In [108]:
installment_risk = (
    df.groupby("installment_rate")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

installment_risk["bad_credit_rate"] = (
    installment_risk["bad_credit"] /
    installment_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

installment_risk["difference_from_overall"] = (
    installment_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print(installment_risk)

In [109]:
plt.figure(figsize=(8, 5))

plt.bar(
    installment_risk.index.astype(str),
    installment_risk["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Historical Bad-Credit Rate by Installment Rate Tier")
plt.xlabel("Installment Rate Tier")
plt.ylabel("Bad-Credit Rate (%)")
plt.legend()

plt.tight_layout()
plt.show()

In [110]:
plt.figure(figsize=(8, 5))

plt.bar(
    installment_risk.index.astype(str),
    installment_risk["applicants"]
)

plt.title("Applicants by Installment Rate Tier")
plt.xlabel("Installment Rate Tier")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

In [111]:
print("Rate change across consecutive tiers (diff):")
print(installment_risk["bad_credit_rate"].diff())

## 26. Default Rate by Present Residence Duration

Present residence duration represents how long the applicant has been living at their current residence, recorded using four ordered tiers. We compare historical bad-credit rates across these tiers to investigate whether residential stability is associated with credit risk.

In [113]:
residence_risk = (
    df.groupby("present_residence_since")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

residence_risk["bad_credit_rate"] = (
    residence_risk["bad_credit"] /
    residence_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

residence_risk["difference_from_overall"] = (
    residence_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

print(residence_risk)

residence_risk["rate_change"] = (
    residence_risk["bad_credit_rate"].diff().round(2)
)

print("\nResidence risk with consecutive rate changes:")
print(residence_risk)

In [114]:
plt.figure(figsize=(8, 5))

plt.bar(
    residence_risk.index.astype(str),
    residence_risk["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Historical Bad-Credit Rate by Residence Duration Tier")
plt.xlabel("Residence Duration Tier")
plt.ylabel("Bad-Credit Rate (%)")
plt.legend()

plt.tight_layout()
plt.show()

In [115]:
plt.figure(figsize=(8, 5))

plt.bar(
    residence_risk.index.astype(str),
    residence_risk["applicants"]
)

plt.title("Applicants by Residence Duration Tier")
plt.xlabel("Residence Duration Tier")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

## 27. Default Rate by Other Debtors / Guarantors

Other debtors or guarantors indicates whether another person is financially associated with the loan as a co-applicant or guarantor. We compare historical bad-credit rates across these categories to understand whether additional financial parties are associated with credit risk.

In [117]:
guarantor_risk = (
    df.groupby("other_debtors_guarantors")
    .agg(
        applicants=("credit_risk", "size"),
        bad_credit=("credit_risk", lambda x: (x == 2).sum())
    )
)

guarantor_risk["bad_credit_rate"] = (
    guarantor_risk["bad_credit"] /
    guarantor_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

guarantor_risk["difference_from_overall"] = (
    guarantor_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

category_names = {
    "A101": "None",
    "A102": "Co-applicant",
    "A103": "Guarantor"
}

guarantor_display = guarantor_risk.copy()

guarantor_display["category_name"] = (
    guarantor_display.index.map(category_names)
)

print(
    guarantor_display[
        [
            "category_name",
            "applicants",
            "bad_credit",
            "bad_credit_rate",
            "difference_from_overall"
        ]
    ]
)

In [118]:
plt.figure(figsize=(9, 5))

plt.bar(
    guarantor_display["category_name"],
    guarantor_display["bad_credit_rate"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

plt.title("Historical Bad-Credit Rate by Other Debtors / Guarantors")
plt.xlabel("Other Debtors / Guarantors")
plt.ylabel("Bad-Credit Rate (%)")
plt.legend()

plt.tight_layout()
plt.show()

In [119]:
plt.figure(figsize=(9, 5))

plt.bar(
    guarantor_display["category_name"],
    guarantor_display["applicants"]
)

plt.title("Applicants by Other Debtors / Guarantors")
plt.xlabel("Other Debtors / Guarantors")
plt.ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

## 28. Default Rate by Other Installment Plans

Other installment plans indicates whether the applicant has other active installment payment liabilities with banks, department stores, or no other installment liabilities. We compare historical bad-credit rates across these categories to investigate whether external installment obligations are associated with credit risk.

In [121]:
installment_plans_risk = (
    df.groupby("other_installment_plans")
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
)

installment_plans_risk["bad_credit_rate"] = (
    installment_plans_risk["bad_credit"] /
    installment_plans_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

installment_plans_risk["difference_from_overall"] = (
    installment_plans_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

plan_names = {
    "A141": "Bank",
    "A142": "Stores",
    "A143": "None"
}

installment_plans_display = installment_plans_risk.copy()
installment_plans_display["category_name"] = (
    installment_plans_display.index.map(plan_names)
)

print("=== Other Installment Plans Summary Table ===")
print(
    installment_plans_display[
        [
            "category_name",
            "applicants",
            "good_credit",
            "bad_credit",
            "bad_credit_rate",
            "difference_from_overall"
        ]
    ]
)

In [122]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

highest_risk_cat = installment_plans_display["bad_credit_rate"].idxmax()
lowest_risk_cat = installment_plans_display["bad_credit_rate"].idxmin()
largest_cat = installment_plans_display["applicants"].idxmax()
smallest_cat = installment_plans_display["applicants"].idxmin()

print(f"Highest-risk Category: {highest_risk_cat} ({plan_names[highest_risk_cat]}) - {installment_plans_display.loc[highest_risk_cat, 'bad_credit_rate']:.2f}%")
print(f"Lowest-risk Category: {lowest_risk_cat} ({plan_names[lowest_risk_cat]}) - {installment_plans_display.loc[lowest_risk_cat, 'bad_credit_rate']:.2f}%")
print(f"Largest Category: {largest_cat} ({plan_names[largest_cat]}) - {installment_plans_display.loc[largest_cat, 'applicants']} applicants")
print(f"Smallest Category: {smallest_cat} ({plan_names[smallest_cat]}) - {installment_plans_display.loc[smallest_cat, 'applicants']} applicants")

print("\n--- Sample Size & Sparsity Analysis ---")
for code, row in installment_plans_display.iterrows():
    pct_share = (row["applicants"] / len(df)) * 100
    print(f"{code} ({row['category_name']}): {row['applicants']} applicants ({pct_share:.1f}% of dataset)")

In [123]:
plt.figure(figsize=(9, 5))

bars = plt.bar(
    installment_plans_display["category_name"],
    installment_plans_display["bad_credit_rate"],
    color=["#3498db", "#e67e22", "#2ecc71"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

# Add percentage values above each bar
for bar in bars:
    height = bar.get_height()
    plt.annotate(
        f"{height:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.title("Historical Bad-Credit Rate by Other Installment Plans")
plt.xlabel("Other Installment Plan Category")
plt.ylabel("Bad-Credit Rate (%)")
plt.ylim(0, max(installment_plans_display["bad_credit_rate"]) + 10)
plt.legend()

plt.tight_layout()
plt.show()

# Applicant volume chart
plt.figure(figsize=(9, 5))

bars_vol = plt.bar(
    installment_plans_display["category_name"],
    installment_plans_display["applicants"],
    color="#34495e"
)

for bar in bars_vol:
    height = bar.get_height()
    plt.annotate(
        f"{int(height)}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.title("Applicants by Other Installment Plans")
plt.xlabel("Other Installment Plan Category")
plt.ylabel("Number of Applicants")
plt.ylim(0, max(installment_plans_display["applicants"]) + 100)

plt.tight_layout()
plt.show()

## 29. Default Rate by Telephone Status

Telephone status indicates whether the applicant has a registered telephone under their name. We compare historical bad-credit rates across telephone categories to investigate whether telephone registration is associated with credit risk in the original dataset.

In [125]:
telephone_risk = (
    df.groupby("telephone")
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
)

telephone_risk["bad_credit_rate"] = (
    telephone_risk["bad_credit"] /
    telephone_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

telephone_risk["difference_from_overall"] = (
    telephone_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

telephone_names = {
    "A191": "None",
    "A192": "Yes - Registered"
}

telephone_display = telephone_risk.copy()
telephone_display["category_name"] = (
    telephone_display.index.map(telephone_names)
)

print("=== Telephone Status Summary Table ===")
print(
    telephone_display[
        [
            "category_name",
            "applicants",
            "good_credit",
            "bad_credit",
            "bad_credit_rate",
            "difference_from_overall"
        ]
    ]
)

In [126]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

highest_risk_cat = telephone_display["bad_credit_rate"].idxmax()
lowest_risk_cat = telephone_display["bad_credit_rate"].idxmin()
largest_cat = telephone_display["applicants"].idxmax()
smallest_cat = telephone_display["applicants"].idxmin()

print(f"Highest-risk Category: {highest_risk_cat} ({telephone_names[highest_risk_cat]}) - {telephone_display.loc[highest_risk_cat, 'bad_credit_rate']:.2f}%")
print(f"Lowest-risk Category: {lowest_risk_cat} ({telephone_names[lowest_risk_cat]}) - {telephone_display.loc[lowest_risk_cat, 'bad_credit_rate']:.2f}%")
print(f"Largest Category: {largest_cat} ({telephone_names[largest_cat]}) - {telephone_display.loc[largest_cat, 'applicants']} applicants")
print(f"Smallest Category: {smallest_cat} ({telephone_names[smallest_cat]}) - {telephone_display.loc[smallest_cat, 'applicants']} applicants")

print("\n--- Sample Size Analysis ---")
for code, row in telephone_display.iterrows():
    pct_share = (row["applicants"] / len(df)) * 100
    print(f"{code} ({row['category_name']}): {row['applicants']} applicants ({pct_share:.1f}% of dataset)")
print("\nBoth categories have sufficient sample sizes for stable descriptive comparison.")

In [127]:
plt.figure(figsize=(8, 5))

bars = plt.bar(
    telephone_display["category_name"],
    telephone_display["bad_credit_rate"],
    color=["#e74c3c", "#2ecc71"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="navy",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

for bar in bars:
    height = bar.get_height()
    plt.annotate(
        f"{height:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

plt.title("Historical Bad-Credit Rate by Telephone Status")
plt.xlabel("Telephone Status")
plt.ylabel("Bad-Credit Rate (%)")
plt.ylim(0, max(telephone_display["bad_credit_rate"]) + 12)
plt.legend()

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))

bars_vol = plt.bar(
    telephone_display["category_name"],
    telephone_display["applicants"],
    color="#7f8c8d"
)

for bar in bars_vol:
    height = bar.get_height()
    plt.annotate(
        f"{int(height)}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

plt.title("Applicants by Telephone Status")
plt.xlabel("Telephone Status")
plt.ylabel("Number of Applicants")
plt.ylim(0, max(telephone_display["applicants"]) + 100)

plt.tight_layout()
plt.show()

## 30. Default Rate by Foreign Worker Status

Foreign worker status records whether the applicant was classified as a foreign worker in the original German Credit Dataset. We analyze historical bad-credit rates across categories **strictly for descriptive EDA and fairness awareness purposes**.

> **Fairness Notice:** `foreign_worker` is a sensitive demographic-related attribute. Any observed difference in default rates reflects historical socioeconomic, employment, documentation, and underwriting disparities in the original dataset — not inherent creditworthiness differences. This feature should **not** be used directly in production credit decisions without thorough governance review.

In [129]:
foreign_worker_risk = (
    df.groupby("foreign_worker")
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
)

foreign_worker_risk["bad_credit_rate"] = (
    foreign_worker_risk["bad_credit"] /
    foreign_worker_risk["applicants"] * 100
).round(2)

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

foreign_worker_risk["difference_from_overall"] = (
    foreign_worker_risk["bad_credit_rate"] -
    overall_bad_rate
).round(2)

foreign_worker_names = {
    "A201": "Yes (Foreign Worker)",
    "A202": "No (Non-Foreign Worker)"
}

foreign_worker_display = foreign_worker_risk.copy()
foreign_worker_display["category_name"] = (
    foreign_worker_display.index.map(foreign_worker_names)
)

print("=== Foreign Worker Status Summary Table ===")
print(
    foreign_worker_display[
        [
            "category_name",
            "applicants",
            "good_credit",
            "bad_credit",
            "bad_credit_rate",
            "difference_from_overall"
        ]
    ]
)

In [130]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

highest_risk_cat = foreign_worker_display["bad_credit_rate"].idxmax()
lowest_risk_cat = foreign_worker_display["bad_credit_rate"].idxmin()
largest_cat = foreign_worker_display["applicants"].idxmax()
smallest_cat = foreign_worker_display["applicants"].idxmin()

print(f"Highest-risk Category: {highest_risk_cat} ({foreign_worker_names[highest_risk_cat]}) - {foreign_worker_display.loc[highest_risk_cat, 'bad_credit_rate']:.2f}%")
print(f"Lowest-risk Category:  {lowest_risk_cat} ({foreign_worker_names[lowest_risk_cat]}) - {foreign_worker_display.loc[lowest_risk_cat, 'bad_credit_rate']:.2f}%")
print(f"Largest Category:      {largest_cat} ({foreign_worker_names[largest_cat]}) - {foreign_worker_display.loc[largest_cat, 'applicants']} applicants")
print(f"Smallest Category:     {smallest_cat} ({foreign_worker_names[smallest_cat]}) - {foreign_worker_display.loc[smallest_cat, 'applicants']} applicants")

print("\n--- Sample Size & Sparsity Analysis ---")
for code, row in foreign_worker_display.iterrows():
    pct_share = (row["applicants"] / len(df)) * 100
    stability = "STABLE" if row["applicants"] >= 50 else "SPARSE - interpret with caution"
    print(f"{code} ({row['category_name']}): {row['applicants']} applicants ({pct_share:.1f}% of dataset) [{stability}]")

In [131]:
plt.figure(figsize=(9, 5))

bars = plt.bar(
    foreign_worker_display["category_name"],
    foreign_worker_display["bad_credit_rate"],
    color=["#e67e22", "#2980b9"]
)

plt.axhline(
    overall_bad_rate,
    linestyle="--",
    color="red",
    label=f"Overall = {overall_bad_rate:.1f}%"
)

for bar in bars:
    height = bar.get_height()
    plt.annotate(
        f"{height:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

plt.title("Historical Bad-Credit Rate by Foreign Worker Status")
plt.xlabel("Foreign Worker Status")
plt.ylabel("Bad-Credit Rate (%)")
plt.ylim(0, max(foreign_worker_display["bad_credit_rate"]) + 15)
plt.legend()

plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))

bars_vol = plt.bar(
    foreign_worker_display["category_name"],
    foreign_worker_display["applicants"],
    color="#7f8c8d"
)

for bar in bars_vol:
    height = bar.get_height()
    plt.annotate(
        f"{int(height)}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

plt.title("Applicants by Foreign Worker Status")
plt.xlabel("Foreign Worker Status")
plt.ylabel("Number of Applicants")
plt.ylim(0, max(foreign_worker_display["applicants"]) + 100)

plt.tight_layout()
plt.show()

## 31. Default Rate by Job and Housing Interaction

In univariate analysis, we examine one feature at a time. In interaction analysis, we examine two features simultaneously to identify patterns that may be hidden when each feature is viewed independently.

Here we analyze the **Job × Housing** interaction:
- **Job** proxies for employment stability and income qualification tier.
- **Housing** proxies for asset ownership and residential stability.

Their combination may identify distinct socioeconomic profiles with different historical default rates — even when neither variable individually shows strong separation. All results represent **descriptive statistical associations**, not causal claims.

In [133]:
import seaborn as sns

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

interaction_risk = (
    df.groupby(["job", "housing"])
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
    .reset_index()
)

interaction_risk["bad_credit_rate"] = (
    interaction_risk["bad_credit"] /
    interaction_risk["applicants"] * 100
).round(2)

interaction_risk["difference_from_overall"] = (
    interaction_risk["bad_credit_rate"] - overall_bad_rate
).round(2)

job_names = {
    "A171": "Unemployed/Unskilled Non-res",
    "A172": "Unskilled Resident",
    "A173": "Skilled Employee/Official",
    "A174": "Management/Self-employed"
}

housing_names = {
    "A151": "Rent",
    "A152": "Own home",
    "A153": "For free"
}

interaction_risk["job_name"] = interaction_risk["job"].map(job_names)
interaction_risk["housing_name"] = interaction_risk["housing"].map(housing_names)

print("=== Job x Housing Interaction — Full Detail Table ===")
print(
    interaction_risk[
        [
            "job", "housing",
            "job_name", "housing_name",
            "applicants", "good_credit", "bad_credit",
            "bad_credit_rate", "difference_from_overall"
        ]
    ].to_string(index=False)
)

In [134]:
pivot_rate = interaction_risk.pivot_table(
    index="job",
    columns="housing",
    values="bad_credit_rate",
    aggfunc="first"
)

pivot_rate.index = [job_names.get(j, j) for j in pivot_rate.index]
pivot_rate.columns = [housing_names.get(h, h) for h in pivot_rate.columns]

print("=== Pivot Table: Bad-Credit Rate (%) by Job x Housing ===")
print(pivot_rate.round(2))

In [135]:
pivot_count = interaction_risk.pivot_table(
    index="job",
    columns="housing",
    values="applicants",
    aggfunc="first"
)

pivot_count.index = [job_names.get(j, j) for j in pivot_count.index]
pivot_count.columns = [housing_names.get(h, h) for h in pivot_count.columns]

print("=== Pivot Table: Applicant Count by Job x Housing ===")
print(pivot_count)

In [136]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    pivot_rate,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=60,
    cbar_kws={"label": "Bad-Credit Rate (%)"},
    ax=ax
)

ax.set_title("Bad-Credit Rate (%) — Job x Housing Interaction", fontsize=13, pad=14)
ax.set_xlabel("Housing Status", fontsize=11)
ax.set_ylabel("Job Category", fontsize=11)
ax.tick_params(axis="x", labelrotation=15)
ax.tick_params(axis="y", labelrotation=0)

plt.tight_layout()
plt.show()

In [137]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

# Highest and lowest risk combinations
highest_idx = interaction_risk["bad_credit_rate"].idxmax()
lowest_idx = interaction_risk["bad_credit_rate"].idxmin()
largest_idx = interaction_risk["applicants"].idxmax()
smallest_idx = interaction_risk["applicants"].idxmin()

h = interaction_risk.loc[highest_idx]
l = interaction_risk.loc[lowest_idx]
lg = interaction_risk.loc[largest_idx]
sm = interaction_risk.loc[smallest_idx]

print(f"Highest-risk combination: {h['job']} ({h['job_name']}) x {h['housing']} ({h['housing_name']}) - {h['bad_credit_rate']:.2f}% ({int(h['applicants'])} applicants)")
print(f"Lowest-risk combination:  {l['job']} ({l['job_name']}) x {l['housing']} ({l['housing_name']}) - {l['bad_credit_rate']:.2f}% ({int(l['applicants'])} applicants)")
print(f"Largest combination:      {lg['job']} ({lg['job_name']}) x {lg['housing']} ({lg['housing_name']}) - {int(lg['applicants'])} applicants")
print(f"Smallest combination:     {sm['job']} ({sm['job_name']}) x {sm['housing']} ({sm['housing_name']}) - {int(sm['applicants'])} applicants")

print("\n--- Sparsity Analysis ---")
sparse_25 = interaction_risk[interaction_risk["applicants"] < 25]
sparse_10 = interaction_risk[interaction_risk["applicants"] < 10]

print(f"Total observed combinations: {len(interaction_risk)} / 12 possible")
print(f"Combinations with N < 25 (sparse): {len(sparse_25)}")
print(f"Combinations with N < 10 (extremely sparse): {len(sparse_10)}")

if len(sparse_25) > 0:
    print("\nSparse combinations (N < 25):")
    for _, row in sparse_25.iterrows():
        flag = "EXTREMELY SPARSE" if row["applicants"] < 10 else "SPARSE"
        print(f"  {row['job']} x {row['housing']}: {int(row['applicants'])} applicants, {row['bad_credit_rate']:.2f}% default rate [{flag}]")

## 32. Default Rate by Checking Account and Savings Account Interaction

Both checking account status and savings account balance are liquidity-related features that showed strong individual separation in univariate analysis. Here, we analyze their **two-way interaction (`status_checking_account` × `savings_account`)** to investigate how combined liquidity cushions relate to historical default rates.

- **Checking Account** captures immediate cash flow and transaction overdraft status.
- **Savings Account** captures accumulated reserves and long-term liquidity buffer.

We analyze whether these two liquidity tiers reinforce each other, identify high-risk compounding vulnerabilities (e.g., negative checking + low savings), or reveal non-linear compounding patterns. All findings represent **historical statistical associations**, not causal claims.

In [139]:
overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

interaction_cs = (
    df.groupby(["status_checking_account", "savings_account"])
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
    .reset_index()
)

interaction_cs["bad_credit_rate"] = (
    interaction_cs["bad_credit"] /
    interaction_cs["applicants"] * 100
).round(2)

interaction_cs["difference_from_overall"] = (
    interaction_cs["bad_credit_rate"] - overall_bad_rate
).round(2)

checking_names = {
    "A11": "< 0 DM",
    "A12": "0 <= balance < 200 DM",
    "A13": ">= 200 DM / Salary assign.",
    "A14": "No checking account"
}

savings_names = {
    "A61": "< 100 DM",
    "A62": "100 <= balance < 500 DM",
    "A63": "500 <= balance < 1000 DM",
    "A64": ">= 1000 DM",
    "A65": "Unknown / No savings"
}

interaction_cs["checking_name"] = interaction_cs["status_checking_account"].map(checking_names)
interaction_cs["savings_name"] = interaction_cs["savings_account"].map(savings_names)

def flag_sample(n):
    if n < 10:
        return "Extremely Sparse (N<10)"
    elif n < 25:
        return "Sparse (N<25)"
    else:
        return "Stable"

interaction_cs["sample_flag"] = interaction_cs["applicants"].apply(flag_sample)

print("=== Checking x Savings Account Interaction — Detailed Table ===")
print(
    interaction_cs[
        [
            "status_checking_account", "checking_name",
            "savings_account", "savings_name",
            "applicants", "good_credit", "bad_credit",
            "bad_credit_rate", "difference_from_overall",
            "sample_flag"
        ]
    ].to_string(index=False)
)

In [140]:
# Checking index order & Savings columns order
checking_order = ["A11", "A12", "A13", "A14"]
savings_order = ["A61", "A62", "A63", "A64", "A65"]

pivot_cs_rate = interaction_cs.pivot_table(
    index="status_checking_account",
    columns="savings_account",
    values="bad_credit_rate",
    aggfunc="first"
).reindex(index=checking_order, columns=savings_order)

pivot_cs_rate.index = [checking_names[c] for c in pivot_cs_rate.index]
pivot_cs_rate.columns = [savings_names[s] for s in pivot_cs_rate.columns]

print("=== Pivot Table: Bad-Credit Rate (%) by Checking x Savings ===")
print(pivot_cs_rate.round(2))

pivot_cs_count = interaction_cs.pivot_table(
    index="status_checking_account",
    columns="savings_account",
    values="applicants",
    aggfunc="first"
).reindex(index=checking_order, columns=savings_order).fillna(0).astype(int)

pivot_cs_count.index = [checking_names[c] for c in pivot_cs_count.index]
pivot_cs_count.columns = [savings_names[s] for s in pivot_cs_count.columns]

print("\n=== Pivot Table: Applicant Counts by Checking x Savings ===")
print(pivot_cs_count)

In [141]:
fig, ax = plt.subplots(figsize=(11, 6))

sns.heatmap(
    pivot_cs_rate,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=70,
    cbar_kws={"label": "Bad-Credit Rate (%)"},
    ax=ax
)

ax.set_title("Bad-Credit Rate (%) — Checking Account x Savings Account Interaction", fontsize=13, pad=14)
ax.set_xlabel("Savings Account Balance", fontsize=11)
ax.set_ylabel("Checking Account Status", fontsize=11)
ax.tick_params(axis="x", labelrotation=20)
ax.tick_params(axis="y", labelrotation=0)

plt.tight_layout()
plt.show()

In [142]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

highest_idx = interaction_cs["bad_credit_rate"].idxmax()
lowest_idx = interaction_cs["bad_credit_rate"].idxmin()
largest_idx = interaction_cs["applicants"].idxmax()
smallest_idx = interaction_cs["applicants"].idxmin()

h = interaction_cs.loc[highest_idx]
l = interaction_cs.loc[lowest_idx]
lg = interaction_cs.loc[largest_idx]
sm = interaction_cs.loc[smallest_idx]

print(f"Highest-risk combination: {h['status_checking_account']} ({h['checking_name']}) x {h['savings_account']} ({h['savings_name']}) - {h['bad_credit_rate']:.2f}% ({int(h['applicants'])} applicants)")
print(f"Lowest-risk combination:  {l['status_checking_account']} ({l['checking_name']}) x {l['savings_account']} ({l['savings_name']}) - {l['bad_credit_rate']:.2f}% ({int(l['applicants'])} applicants)")
print(f"Largest combination:      {lg['status_checking_account']} ({lg['checking_name']}) x {lg['savings_account']} ({lg['savings_name']}) - {int(lg['applicants'])} applicants")
print(f"Smallest combination:     {sm['status_checking_account']} ({sm['checking_name']}) x {sm['savings_account']} ({sm['savings_name']}) - {int(sm['applicants'])} applicants")

print("\n--- Sample Size & Sparsity Breakdown ---")
sparse_25 = interaction_cs[(interaction_cs["applicants"] < 25) & (interaction_cs["applicants"] >= 10)]
sparse_10 = interaction_cs[interaction_cs["applicants"] < 10]
stable = interaction_cs[interaction_cs["applicants"] >= 25]

print(f"Total possible combinations: 20 (4 checking x 5 savings)")
print(f"Total observed combinations: {len(interaction_cs)}")
print(f"Stable combinations (N >= 25): {len(stable)}")
print(f"Sparse combinations (10 <= N < 25): {len(sparse_25)}")
print(f"Extremely sparse combinations (N < 10): {len(sparse_10)}")

print("\nExtremely sparse combinations (N < 10):")
for _, row in sparse_10.iterrows():
    print(f"  {row['status_checking_account']} x {row['savings_account']}: {int(row['applicants'])} applicants, {row['bad_credit_rate']:.2f}% default rate")

print("\nSparse combinations (10 <= N < 25):")
for _, row in sparse_25.iterrows():
    print(f"  {row['status_checking_account']} x {row['savings_account']}: {int(row['applicants'])} applicants, {row['bad_credit_rate']:.2f}% default rate")

## 33. Default Rate by Loan Duration and Credit Amount Interaction

In univariate analysis, both **Loan Duration** and **Credit Amount** showed strong positive associations with default risk:
- Longer repayment horizons increase cumulative exposure time and repayment uncertainty.
- Larger credit amounts increase absolute repayment burden.

Because duration and credit amount are strongly correlated in credit portfolios (large loans typically require extended loan terms), analyzing their **5 × 4 two-way interaction** allows us to:
1. Disentangle whether duration separates risk independently within credit amount tiers.
2. Investigate whether high-duration + high-amount loans compound into an extreme-risk portfolio segment.
3. Identify structural empty regions where certain loan terms do not exist.

All findings represent **historical statistical associations**, not direct causal relationships.

In [144]:
import numpy as np

# Define duration brackets
duration_bins = [0, 12, 24, 36, 48, 100]
duration_labels = ["4–12 months", "13–24 months", "25–36 months", "37–48 months", "49+ months"]
duration_bracket = pd.cut(df["duration_in_months"], bins=duration_bins, labels=duration_labels, right=True)

# Calculate dynamic credit amount quartiles
amount_quartiles, q_edges = pd.qcut(
    df["credit_amount"],
    q=4,
    labels=["Q1 Lowest", "Q2", "Q3", "Q4 Highest"],
    retbins=True
)

print(f"Credit Amount Quartile Boundaries:")
print(f"  Q1 Lowest:  {q_edges[0]:.0f} to {q_edges[1]:.0f} DM")
print(f"  Q2:         {q_edges[1]:.0f} to {q_edges[2]:.0f} DM")
print(f"  Q3:         {q_edges[2]:.0f} to {q_edges[3]:.0f} DM")
print(f"  Q4 Highest: {q_edges[3]:.0f} to {q_edges[4]:.0f} DM\n")

temp_df = pd.DataFrame({
    "duration_bracket": duration_bracket,
    "credit_amount_quartile": amount_quartiles,
    "credit_risk": df["credit_risk"]
})

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

interaction_da = (
    temp_df.groupby(["duration_bracket", "credit_amount_quartile"], observed=False)
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
    .reset_index()
)

interaction_da["bad_credit_rate"] = np.where(
    interaction_da["applicants"] > 0,
    (interaction_da["bad_credit"] / interaction_da["applicants"] * 100).round(2),
    np.nan
)

interaction_da["difference_from_overall"] = np.where(
    interaction_da["applicants"] > 0,
    (interaction_da["bad_credit_rate"] - overall_bad_rate).round(2),
    np.nan
)

def flag_sample(n):
    if n == 0:
        return "Unobserved (N=0)"
    elif n < 10:
        return "Extremely Sparse (N<10)"
    elif n < 25:
        return "Sparse (N<25)"
    else:
        return "Stable"

interaction_da["sample_flag"] = interaction_da["applicants"].apply(flag_sample)

print("=== Duration x Credit Amount Interaction — Detailed Table ===")
print(
    interaction_da[
        [
            "duration_bracket", "credit_amount_quartile",
            "applicants", "good_credit", "bad_credit",
            "bad_credit_rate", "difference_from_overall",
            "sample_flag"
        ]
    ].to_string(index=False)
)

In [145]:
pivot_da_rate = interaction_da.pivot_table(
    index="duration_bracket",
    columns="credit_amount_quartile",
    values="bad_credit_rate",
    observed=False
)

print("=== Pivot Table: Bad-Credit Rate (%) by Duration x Credit Amount ===")
print(pivot_da_rate.round(2))

pivot_da_count = interaction_da.pivot_table(
    index="duration_bracket",
    columns="credit_amount_quartile",
    values="applicants",
    observed=False
).fillna(0).astype(int)

print("\n=== Pivot Table: Applicant Counts by Duration x Credit Amount ===")
print(pivot_da_count)

In [146]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    pivot_da_rate,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=70,
    mask=pivot_da_count == 0,
    cbar_kws={"label": "Bad-Credit Rate (%)"},
    ax=ax
)

ax.set_title("Bad-Credit Rate (%) — Duration x Credit Amount Interaction", fontsize=13, pad=14)
ax.set_xlabel("Credit Amount Quartile", fontsize=11)
ax.set_ylabel("Loan Duration Bracket", fontsize=11)
ax.tick_params(axis="x", labelrotation=0)
ax.tick_params(axis="y", labelrotation=0)

plt.tight_layout()
plt.show()

In [147]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

observed_cells = interaction_da[interaction_da["applicants"] > 0]
stable_cells = interaction_da[interaction_da["applicants"] >= 25]
sparse_cells = interaction_da[(interaction_da["applicants"] < 25) & (interaction_da["applicants"] >= 10)]
ext_sparse_cells = interaction_da[(interaction_da["applicants"] < 10) & (interaction_da["applicants"] > 0)]
unobserved_cells = interaction_da[interaction_da["applicants"] == 0]

highest_raw = observed_cells.loc[observed_cells["bad_credit_rate"].idxmax()]
lowest_raw = observed_cells.loc[observed_cells["bad_credit_rate"].idxmin()]
highest_stable = stable_cells.loc[stable_cells["bad_credit_rate"].idxmax()]
lowest_stable = stable_cells.loc[stable_cells["bad_credit_rate"].idxmin()]
largest_cell = observed_cells.loc[observed_cells["applicants"].idxmax()]
smallest_obs = observed_cells.loc[observed_cells["applicants"].idxmin()]

print(f"Highest-risk Interaction (Empirical):        {highest_raw['duration_bracket']} x {highest_raw['credit_amount_quartile']} - {highest_raw['bad_credit_rate']:.2f}% ({int(highest_raw['applicants'])} app)")
print(f"Lowest-risk Interaction (Empirical):         {lowest_raw['duration_bracket']} x {lowest_raw['credit_amount_quartile']} - {lowest_raw['bad_credit_rate']:.2f}% ({int(lowest_raw['applicants'])} app)")
print(f"Highest-risk Stable Interaction (N >= 25):  {highest_stable['duration_bracket']} x {highest_stable['credit_amount_quartile']} - {highest_stable['bad_credit_rate']:.2f}% ({int(highest_stable['applicants'])} app)")
print(f"Lowest-risk Stable Interaction (N >= 25):   {lowest_stable['duration_bracket']} x {lowest_stable['credit_amount_quartile']} - {lowest_stable['bad_credit_rate']:.2f}% ({int(lowest_stable['applicants'])} app)")
print(f"Largest Combination:                         {largest_cell['duration_bracket']} x {largest_cell['credit_amount_quartile']} - {int(largest_cell['applicants'])} applicants")
print(f"Smallest Observed Combination:               {smallest_obs['duration_bracket']} x {smallest_obs['credit_amount_quartile']} - {int(smallest_obs['applicants'])} applicants")

print("\n--- Sample Size Breakdown ---")
print(f"Total possible combinations: 20 (5 duration brackets x 4 amount quartiles)")
print(f"Observed combinations:       {len(observed_cells)}")
print(f"Unobserved combinations:     {len(unobserved_cells)}")
print(f"Stable cells (N >= 25):      {len(stable_cells)}")
print(f"Sparse cells (10 <= N < 25): {len(sparse_cells)}")
print(f"Extremely sparse (N < 10):   {len(ext_sparse_cells)}")

print("\n--- Within-Quartile and Within-Duration Gradient Observations ---")
print("1. In Q4 Highest: Default rates rise steeply with duration: 13–24 mo (33.75%) -> 25–36 mo (41.25%) -> 37–48 mo (50.88%) -> 49+ mo (50.00%).")
print("2. In 4–12 mo: Low-to-moderate amounts (Q2 & Q3) show very low default rates (14.91% and 11.11%).")
print("3. Structural unobserved cells: No applicants in 49+ months for Q1, Q2, Q3.")

## 34. Default Rate by Age Group and Employment Duration Interaction

In univariate analysis:
- **Age** (Section 16) showed younger borrowers (18–25) carrying substantially higher default risk (~42%) than mid-career and older borrowers (~25–29%).
- **Employment Duration** (Section 17) showed applicants with short tenure (<1 year) carrying higher default risk (~41%) compared to tenured workers ($\ge 7$ years, ~25%).

Here we analyze their **5 × 5 two-way interaction (`age_group` × `present_employment_since`)** to investigate:
1. Compounding instability (e.g., young applicants with short employment history).
2. Whether employment stability compensates for younger age (or vice versa).
3. Underwriting selection patterns in unusual cohorts (such as unemployed older applicants).

All findings represent **descriptive historical statistical associations**, not causal claims.

In [149]:
import numpy as np

# Age group bins
age_bins = [17, 25, 35, 45, 55, 120]
age_labels = ["18–25", "26–35", "36–45", "46–55", "56+"]
age_group = pd.cut(df["age_in_years"], bins=age_bins, labels=age_labels, right=True)

emp_names = {
    "A71": "Unemployed",
    "A72": "< 1 year",
    "A73": "1–4 years",
    "A74": "4–7 years",
    "A75": ">= 7 years"
}
emp_order = ["A71", "A72", "A73", "A74", "A75"]

temp_ae = pd.DataFrame({
    "age_group": age_group,
    "employment": df["present_employment_since"],
    "credit_risk": df["credit_risk"]
})

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

interaction_ae = (
    temp_ae.groupby(["age_group", "employment"], observed=False)
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
    .reset_index()
)

interaction_ae["employment_name"] = interaction_ae["employment"].map(emp_names)
interaction_ae["bad_credit_rate"] = np.where(
    interaction_ae["applicants"] > 0,
    (interaction_ae["bad_credit"] / interaction_ae["applicants"] * 100).round(2),
    np.nan
)
interaction_ae["difference_from_overall"] = np.where(
    interaction_ae["applicants"] > 0,
    (interaction_ae["bad_credit_rate"] - overall_bad_rate).round(2),
    np.nan
)

def flag_sample(n):
    if n == 0:
        return "Unobserved (N=0)"
    elif n < 10:
        return "Extremely Sparse (N<10)"
    elif n < 25:
        return "Sparse (N<25)"
    else:
        return "Stable"

interaction_ae["sample_flag"] = interaction_ae["applicants"].apply(flag_sample)

print("=== Age Group x Employment Duration Interaction — Detailed Table ===")
print(
    interaction_ae[
        [
            "age_group", "employment", "employment_name",
            "applicants", "good_credit", "bad_credit",
            "bad_credit_rate", "difference_from_overall",
            "sample_flag"
        ]
    ].to_string(index=False)
)

In [150]:
pivot_ae_rate = interaction_ae.pivot_table(
    index="age_group",
    columns="employment",
    values="bad_credit_rate",
    observed=False
).reindex(index=age_labels, columns=emp_order)
pivot_ae_rate.columns = [emp_names[c] for c in pivot_ae_rate.columns]

print("=== Pivot Table: Bad-Credit Rate (%) by Age Group x Employment Duration ===")
print(pivot_ae_rate.round(2))

pivot_ae_count = interaction_ae.pivot_table(
    index="age_group",
    columns="employment",
    values="applicants",
    observed=False
).reindex(index=age_labels, columns=emp_order).fillna(0).astype(int)
pivot_ae_count.columns = [emp_names[c] for c in pivot_ae_count.columns]

print("\n=== Pivot Table: Applicant Counts by Age Group x Employment Duration ===")
print(pivot_ae_count)

In [151]:
fig, ax = plt.subplots(figsize=(11, 6))

sns.heatmap(
    pivot_ae_rate,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=70,
    mask=pivot_ae_count == 0,
    cbar_kws={"label": "Bad-Credit Rate (%)"},
    ax=ax
)

ax.set_title("Bad-Credit Rate (%) — Age Group x Employment Duration Interaction", fontsize=13, pad=14)
ax.set_xlabel("Present Employment Duration", fontsize=11)
ax.set_ylabel("Age Group", fontsize=11)
ax.tick_params(axis="x", labelrotation=15)
ax.tick_params(axis="y", labelrotation=0)

plt.tight_layout()
plt.show()

In [152]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

observed_ae = interaction_ae[interaction_ae["applicants"] > 0]
stable_ae = interaction_ae[interaction_ae["applicants"] >= 25]
sparse_ae = interaction_ae[(interaction_ae["applicants"] < 25) & (interaction_ae["applicants"] >= 10)]
ext_sparse_ae = interaction_ae[(interaction_ae["applicants"] < 10) & (interaction_ae["applicants"] > 0)]

highest_raw = observed_ae.loc[observed_ae["bad_credit_rate"].idxmax()]
lowest_raw = observed_ae.loc[observed_ae["bad_credit_rate"].idxmin()]
highest_stable = stable_ae.loc[stable_ae["bad_credit_rate"].idxmax()]
lowest_stable = stable_ae.loc[stable_ae["bad_credit_rate"].idxmin()]
largest_cell = observed_ae.loc[observed_ae["applicants"].idxmax()]
smallest_obs = observed_ae.loc[observed_ae["applicants"].idxmin()]

print(f"Highest-risk Interaction (Empirical):        {highest_raw['age_group']} x {highest_raw['employment']} ({highest_raw['employment_name']}) - {highest_raw['bad_credit_rate']:.2f}% ({int(highest_raw['applicants'])} app)")
print(f"Lowest-risk Interaction (Empirical):         {lowest_raw['age_group']} x {lowest_raw['employment']} ({lowest_raw['employment_name']}) - {lowest_raw['bad_credit_rate']:.2f}% ({int(lowest_raw['applicants'])} app)")
print(f"Highest-risk Stable Interaction (N >= 25):  {highest_stable['age_group']} x {highest_stable['employment']} ({highest_stable['employment_name']}) - {highest_stable['bad_credit_rate']:.2f}% ({int(highest_stable['applicants'])} app)")
print(f"Lowest-risk Stable Interaction (N >= 25):   {lowest_stable['age_group']} x {lowest_stable['employment']} ({lowest_stable['employment_name']}) - {lowest_stable['bad_credit_rate']:.2f}% ({int(lowest_stable['applicants'])} app)")
print(f"Largest Combination:                         {largest_cell['age_group']} x {largest_cell['employment']} ({largest_cell['employment_name']}) - {int(largest_cell['applicants'])} applicants")
print(f"Smallest Observed Combination:               {smallest_obs['age_group']} x {smallest_obs['employment']} ({smallest_obs['employment_name']}) - {int(smallest_obs['applicants'])} applicants")

print("\n--- Sample Size Breakdown ---")
print(f"Total possible combinations: 25 (5 age groups x 5 employment tiers)")
print(f"Observed combinations:       {len(observed_ae)}")
print(f"Stable cells (N >= 25):      {len(stable_ae)}")
print(f"Sparse cells (10 <= N < 25): {len(sparse_ae)}")
print(f"Extremely sparse (N < 10):   {len(ext_sparse_ae)}")

print("\n--- Key Gradient & Underwriting Observations ---")
print("1. Young + Short Tenure: 18–25 applicants with < 1 year tenure exhibit a 48.15% default rate across 54 applicants.")
print("2. Tenure Separates within Age: For 26–35 applicants, < 1 year tenure has 40.48% default rate vs 22.78% for 4–7 years.")
print("3. Age Separates within Tenure: For < 1 year tenure, default rate drops from 48.15% (18–25) to 40.48% (26–35) to 22.73% (36–45).")
print("4. Unemployed Anomaly: Older unemployed applicants (56+) show 0.00% defaults (N=11), reflecting asset-rich/pension selection bias.")

## 35. Default Rate by Savings Account and Employment Duration Interaction

In previous univariate analyses:
- **Savings Account** (Section 21) showed that liquid reserves strongly differentiate credit risk, with $< 100$ DM savings having a ~36% default rate compared to ~12.5% for $\ge 1,000$ DM.
- **Employment Duration** (Section 17) demonstrated that career stability reduces default risk from ~41% (<1 year) down to ~22–25% ($\ge 4$ years).

Here we analyze their **5 × 5 two-way interaction (`savings_account` × `present_employment_since`)** to investigate:
1. Compounding vulnerability when low liquidity meets short employment tenure.
2. Whether higher savings cushions protect against job transition risk.
3. Underwriting selection and liquidity profiles in unemployed or sparse cohorts.

All findings represent **historical statistical associations**, not causal claims.

In [154]:
import numpy as np

savings_names = {
    "A61": "< 100 DM",
    "A62": "100 <= balance < 500 DM",
    "A63": "500 <= balance < 1000 DM",
    "A64": ">= 1000 DM",
    "A65": "Unknown / No savings"
}
savings_order = ["A61", "A62", "A63", "A64", "A65"]

emp_names = {
    "A71": "Unemployed",
    "A72": "< 1 year",
    "A73": "1–4 years",
    "A74": "4–7 years",
    "A75": ">= 7 years"
}
emp_order = ["A71", "A72", "A73", "A74", "A75"]

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

interaction_se = (
    df.groupby(["savings_account", "present_employment_since"], observed=False)
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
    .reset_index()
)

interaction_se["savings_name"] = interaction_se["savings_account"].map(savings_names)
interaction_se["employment_name"] = interaction_se["present_employment_since"].map(emp_names)

interaction_se["bad_credit_rate"] = np.where(
    interaction_se["applicants"] > 0,
    (interaction_se["bad_credit"] / interaction_se["applicants"] * 100).round(2),
    np.nan
)
interaction_se["difference_from_overall"] = np.where(
    interaction_se["applicants"] > 0,
    (interaction_se["bad_credit_rate"] - overall_bad_rate).round(2),
    np.nan
)

def flag_sample(n):
    if n == 0:
        return "Unobserved (N=0)"
    elif n < 10:
        return "Extremely Sparse (N<10)"
    elif n < 25:
        return "Sparse (N<25)"
    else:
        return "Stable"

interaction_se["sample_flag"] = interaction_se["applicants"].apply(flag_sample)

print("=== Savings Account x Employment Duration Interaction — Detailed Table ===")
print(
    interaction_se[
        [
            "savings_account", "savings_name",
            "present_employment_since", "employment_name",
            "applicants", "good_credit", "bad_credit",
            "bad_credit_rate", "difference_from_overall",
            "sample_flag"
        ]
    ].to_string(index=False)
)

In [155]:
pivot_se_rate = interaction_se.pivot_table(
    index="savings_account",
    columns="present_employment_since",
    values="bad_credit_rate",
    observed=False
).reindex(index=savings_order, columns=emp_order)
pivot_se_rate.index = [savings_names[s] for s in pivot_se_rate.index]
pivot_se_rate.columns = [emp_names[e] for e in pivot_se_rate.columns]

print("=== Pivot Table: Bad-Credit Rate (%) by Savings x Employment ===")
print(pivot_se_rate.round(2))

pivot_se_count = interaction_se.pivot_table(
    index="savings_account",
    columns="present_employment_since",
    values="applicants",
    observed=False
).reindex(index=savings_order, columns=emp_order).fillna(0).astype(int)
pivot_se_count.index = [savings_names[s] for s in pivot_se_count.index]
pivot_se_count.columns = [emp_names[e] for e in pivot_se_count.columns]

print("\n=== Pivot Table: Applicant Counts by Savings x Employment ===")
print(pivot_se_count)

In [156]:
fig, ax = plt.subplots(figsize=(11, 6))

sns.heatmap(
    pivot_se_rate,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=70,
    mask=pivot_se_count == 0,
    cbar_kws={"label": "Bad-Credit Rate (%)"},
    ax=ax
)

ax.set_title("Bad-Credit Rate (%) — Savings Account x Employment Duration Interaction", fontsize=13, pad=14)
ax.set_xlabel("Present Employment Duration", fontsize=11)
ax.set_ylabel("Savings Account Balance", fontsize=11)
ax.tick_params(axis="x", labelrotation=15)
ax.tick_params(axis="y", labelrotation=0)

plt.tight_layout()
plt.show()

In [157]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

observed_se = interaction_se[interaction_se["applicants"] > 0]
stable_se = interaction_se[interaction_se["applicants"] >= 25]
sparse_se = interaction_se[(interaction_se["applicants"] < 25) & (interaction_se["applicants"] >= 10)]
ext_sparse_se = interaction_se[(interaction_se["applicants"] < 10) & (interaction_se["applicants"] > 0)]
unobserved_se = interaction_se[interaction_se["applicants"] == 0]

highest_raw = observed_se.loc[observed_se["bad_credit_rate"].idxmax()]
lowest_raw = observed_se.loc[observed_se["bad_credit_rate"].idxmin()]
highest_stable = stable_se.loc[stable_se["bad_credit_rate"].idxmax()]
lowest_stable = stable_se.loc[stable_se["bad_credit_rate"].idxmin()]
largest_cell = observed_se.loc[observed_se["applicants"].idxmax()]
smallest_obs = observed_se.loc[observed_se["applicants"].idxmin()]

print(f"Highest-risk Interaction (Empirical):        {highest_raw['savings_account']} ({highest_raw['savings_name']}) x {highest_raw['present_employment_since']} ({highest_raw['employment_name']}) - {highest_raw['bad_credit_rate']:.2f}% ({int(highest_raw['applicants'])} app)")
print(f"Lowest-risk Interaction (Empirical):         {lowest_raw['savings_account']} ({lowest_raw['savings_name']}) x {lowest_raw['present_employment_since']} ({lowest_raw['employment_name']}) - {lowest_raw['bad_credit_rate']:.2f}% ({int(lowest_raw['applicants'])} app)")
print(f"Highest-risk Stable Interaction (N >= 25):  {highest_stable['savings_account']} ({highest_stable['savings_name']}) x {highest_stable['present_employment_since']} ({highest_stable['employment_name']}) - {highest_stable['bad_credit_rate']:.2f}% ({int(highest_stable['applicants'])} app)")
print(f"Lowest-risk Stable Interaction (N >= 25):   {lowest_stable['savings_account']} ({lowest_stable['savings_name']}) x {lowest_stable['present_employment_since']} ({lowest_stable['employment_name']}) - {lowest_stable['bad_credit_rate']:.2f}% ({int(lowest_stable['applicants'])} app)")
print(f"Largest Combination:                         {largest_cell['savings_account']} ({largest_cell['savings_name']}) x {largest_cell['present_employment_since']} ({largest_cell['employment_name']}) - {int(largest_cell['applicants'])} applicants")
print(f"Smallest Observed Combination:               {smallest_obs['savings_account']} ({smallest_obs['savings_name']}) x {smallest_obs['present_employment_since']} ({smallest_obs['employment_name']}) - {int(smallest_obs['applicants'])} applicants")

print("\n--- Sample Size Breakdown ---")
print(f"Total possible combinations: 25 (5 savings tiers x 5 employment tiers)")
print(f"Observed combinations:       {len(observed_se)}")
print(f"Structurally unobserved:     {len(unobserved_se)} (A64 >=1000 DM x A71 Unemployed)")
print(f"Stable cells (N >= 25):      {len(stable_se)}")
print(f"Sparse cells (10 <= N < 25): {len(sparse_se)}")
print(f"Extremely sparse (N < 10):   {len(ext_sparse_se)}")

print("\n--- Key Gradient & Resilience Observations ---")
print("1. Low Savings + Short Tenure: A61 (< 100 DM) x A72 (< 1 year) shows a 44.17% default rate across 120 applicants.")
print("2. Savings Cushion Mitigates Short Tenure: For < 1 year tenure, default rate drops from 44.17% (A61) to 14.29% (A64) and 13.04% (A65).")
print("3. Employment Tenure Gradients within Savings: In A62 (100–500 DM), default rate drops from 58.82% (< 1 year) to 16.67% (4–7 years).")
print("4. Structurally Empty: No applicants with >=1000 DM savings were unemployed (A64 x A71).")

## 36. Default Rate by Credit History and Checking Account Interaction

In previous univariate analyses:
- **Credit History** (Section 14) showed a seemingly paradoxical pattern where "critical accounts / other credits existing" (`A34`) had a lower default rate (~17%) than "no credits taken / all paid duly" (`A30`, ~62.5%), heavily driven by underwriting selection bias.
- **Checking Account Status** (Section 15) proved to be the single strongest univariate risk discriminator in the dataset, with `< 0 DM` (`A11`) showing a ~49% default rate compared to ~11.7% for `No checking account` (`A14`).

Here we analyze their **5 × 4 two-way interaction (`credit_history` × `status_checking_account`)** to investigate:
1. Whether current liquidity stress (`Checking Account`) overrides past repayment reputation (`Credit History`).
2. Whether the apparent anomalies in credit history resolve once we control for current account stress.
3. How historical reputation interacts with present cash-flow liquidity.

All findings represent **descriptive historical statistical associations**, not causal claims.

In [159]:
import numpy as np

history_names = {
    "A30": "No credits / All paid",
    "A31": "All paid back duly",
    "A32": "Existing paid duly",
    "A33": "Delay in past",
    "A34": "Critical account / Other existing"
}
history_order = ["A30", "A31", "A32", "A33", "A34"]

checking_names = {
    "A11": "< 0 DM",
    "A12": "0 <= balance < 200 DM",
    "A13": ">= 200 DM / Salary assign.",
    "A14": "No checking account"
}
checking_order = ["A11", "A12", "A13", "A14"]

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

interaction_hc = (
    df.groupby(["credit_history", "status_checking_account"], observed=False)
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
    .reset_index()
)

interaction_hc["history_name"] = interaction_hc["credit_history"].map(history_names)
interaction_hc["checking_name"] = interaction_hc["status_checking_account"].map(checking_names)

interaction_hc["bad_credit_rate"] = np.where(
    interaction_hc["applicants"] > 0,
    (interaction_hc["bad_credit"] / interaction_hc["applicants"] * 100).round(2),
    np.nan
)
interaction_hc["difference_from_overall"] = np.where(
    interaction_hc["applicants"] > 0,
    (interaction_hc["bad_credit_rate"] - overall_bad_rate).round(2),
    np.nan
)

def flag_sample(n):
    if n == 0:
        return "Unobserved (N=0)"
    elif n < 10:
        return "Extremely Sparse (N<10)"
    elif n < 25:
        return "Sparse (N<25)"
    else:
        return "Stable"

interaction_hc["sample_flag"] = interaction_hc["applicants"].apply(flag_sample)

print("=== Credit History x Checking Account Interaction — Detailed Table ===")
print(
    interaction_hc[
        [
            "credit_history", "history_name",
            "status_checking_account", "checking_name",
            "applicants", "good_credit", "bad_credit",
            "bad_credit_rate", "difference_from_overall",
            "sample_flag"
        ]
    ].to_string(index=False)
)

In [160]:
pivot_hc_rate = interaction_hc.pivot_table(
    index="credit_history",
    columns="status_checking_account",
    values="bad_credit_rate",
    observed=False
).reindex(index=history_order, columns=checking_order)
pivot_hc_rate.index = [history_names[h] for h in pivot_hc_rate.index]
pivot_hc_rate.columns = [checking_names[c] for c in pivot_hc_rate.columns]

print("=== Pivot Table: Bad-Credit Rate (%) by Credit History x Checking Account ===")
print(pivot_hc_rate.round(2))

pivot_hc_count = interaction_hc.pivot_table(
    index="credit_history",
    columns="status_checking_account",
    values="applicants",
    observed=False
).reindex(index=history_order, columns=checking_order).fillna(0).astype(int)
pivot_hc_count.index = [history_names[h] for h in pivot_hc_count.index]
pivot_hc_count.columns = [checking_names[c] for c in pivot_hc_count.columns]

print("\n=== Pivot Table: Applicant Counts by Credit History x Checking Account ===")
print(pivot_hc_count)

In [161]:
fig, ax = plt.subplots(figsize=(11, 6))

sns.heatmap(
    pivot_hc_rate,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=80,
    mask=pivot_hc_count == 0,
    cbar_kws={"label": "Bad-Credit Rate (%)"},
    ax=ax
)

ax.set_title("Bad-Credit Rate (%) — Credit History x Checking Account Interaction", fontsize=13, pad=14)
ax.set_xlabel("Checking Account Status", fontsize=11)
ax.set_ylabel("Credit History Category", fontsize=11)
ax.tick_params(axis="x", labelrotation=15)
ax.tick_params(axis="y", labelrotation=0)

plt.tight_layout()
plt.show()

In [162]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

observed_hc = interaction_hc[interaction_hc["applicants"] > 0]
stable_hc = interaction_hc[interaction_hc["applicants"] >= 25]
sparse_hc = interaction_hc[(interaction_hc["applicants"] < 25) & (interaction_hc["applicants"] >= 10)]
ext_sparse_hc = interaction_hc[(interaction_hc["applicants"] < 10) & (interaction_hc["applicants"] > 0)]
unobserved_hc = interaction_hc[interaction_hc["applicants"] == 0]

highest_raw = observed_hc.loc[observed_hc["bad_credit_rate"].idxmax()]
lowest_raw = observed_hc.loc[observed_hc["bad_credit_rate"].idxmin()]
highest_stable = stable_hc.loc[stable_hc["bad_credit_rate"].idxmax()]
lowest_stable = stable_hc.loc[stable_hc["bad_credit_rate"].idxmin()]
largest_cell = observed_hc.loc[observed_hc["applicants"].idxmax()]
smallest_obs = observed_hc.loc[observed_hc["applicants"].idxmin()]

print(f"Highest-risk Interaction (Empirical):        {highest_raw['credit_history']} ({highest_raw['history_name']}) x {highest_raw['status_checking_account']} ({highest_raw['checking_name']}) - {highest_raw['bad_credit_rate']:.2f}% ({int(highest_raw['applicants'])} app)")
print(f"Lowest-risk Interaction (Empirical):         {lowest_raw['credit_history']} ({lowest_raw['history_name']}) x {lowest_raw['status_checking_account']} ({lowest_raw['checking_name']}) - {lowest_raw['bad_credit_rate']:.2f}% ({int(lowest_raw['applicants'])} app)")
print(f"Highest-risk Stable Interaction (N >= 25):  {highest_stable['credit_history']} ({highest_stable['history_name']}) x {highest_stable['status_checking_account']} ({highest_stable['checking_name']}) - {highest_stable['bad_credit_rate']:.2f}% ({int(highest_stable['applicants'])} app)")
print(f"Lowest-risk Stable Interaction (N >= 25):   {lowest_stable['credit_history']} ({lowest_stable['history_name']}) x {lowest_stable['status_checking_account']} ({lowest_stable['checking_name']}) - {lowest_stable['bad_credit_rate']:.2f}% ({int(lowest_stable['applicants'])} app)")
print(f"Largest Combination:                         {largest_cell['credit_history']} ({largest_cell['history_name']}) x {largest_cell['status_checking_account']} ({largest_cell['checking_name']}) - {int(largest_cell['applicants'])} applicants")
print(f"Smallest Observed Combination:               {smallest_obs['credit_history']} ({smallest_obs['history_name']}) x {smallest_obs['status_checking_account']} ({smallest_obs['checking_name']}) - {int(smallest_obs['applicants'])} applicants")

print("\n--- Sample Size Breakdown ---")
print(f"Total possible combinations: 20 (5 credit history tiers x 4 checking tiers)")
print(f"Observed combinations:       {len(observed_hc)}")
print(f"Structurally unobserved:     {len(unobserved_hc)}")
print(f"Stable cells (N >= 25):      {len(stable_hc)}")
print(f"Sparse cells (10 <= N < 25): {len(sparse_hc)}")
print(f"Extremely sparse (N < 10):   {len(ext_sparse_hc)}")

print("\n--- Key Gradient & Selection Observations ---")
print("1. Liquidity Stress Dominates Reputation: Clean past history (A30/A31) combined with < 0 DM checking (A11) suffers 72–77% default rates.")
print("2. A14 (No Checking) Low Risk Across All History: Default rates in A14 remain between 6.54% and 28.57%.")
print("3. Critical Account (A34) x No Checking (A14): Exhibits an exceptionally low 6.54% default rate across 153 applicants, driven by underwriting selection bias.")
print("4. Explaining the A30 Paradox: 77.5% of A30 applicants (31/40) were in checking stress (A11/A12), driving its high univariate rate.")

## 37. Default Rate by Credit Amount and Installment Rate Interaction

In univariate analysis:
- **Credit Amount** (Section 13) showed that large loan balances carry substantially higher default risk (~39% in the upper quartile).
- **Installment Rate** (Section 25) demonstrated that higher debt-service burden relative to disposable income increases default rates from ~25% (Tier 1) to ~33% (Tier 4).

Here we analyze their **4 × 4 two-way interaction (`credit_amount_quartile` × `installment_rate`)** to investigate:
1. Whether debt-service pressure (`installment_rate`) compounds with high loan balance (`credit_amount`).
2. Whether low installment burden protects borrowers taking large credit amounts.
3. How cash-flow affordability interacts with absolute balance exposure.

All findings represent **descriptive historical statistical associations**, not direct causal claims.

In [164]:
import numpy as np

# Calculate dynamic credit amount quartiles (same methodology as Section 33)
amount_quartiles, q_edges = pd.qcut(
    df["credit_amount"],
    q=4,
    labels=["Q1 Lowest", "Q2", "Q3", "Q4 Highest"],
    retbins=True
)

print("Credit Amount Quartile Boundaries:")
print(f"  Q1 Lowest:  {q_edges[0]:.0f} to {q_edges[1]:.0f} DM")
print(f"  Q2:         {q_edges[1]:.0f} to {q_edges[2]:.0f} DM")
print(f"  Q3:         {q_edges[2]:.0f} to {q_edges[3]:.0f} DM")
print(f"  Q4 Highest: {q_edges[3]:.0f} to {q_edges[4]:.0f} DM\n")

temp_ai = pd.DataFrame({
    "credit_amount_quartile": amount_quartiles,
    "installment_rate": df["installment_rate"].astype(str),
    "credit_risk": df["credit_risk"]
})

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

interaction_ai = (
    temp_ai.groupby(["credit_amount_quartile", "installment_rate"], observed=False)
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
    .reset_index()
)

interaction_ai["bad_credit_rate"] = np.where(
    interaction_ai["applicants"] > 0,
    (interaction_ai["bad_credit"] / interaction_ai["applicants"] * 100).round(2),
    np.nan
)
interaction_ai["difference_from_overall"] = np.where(
    interaction_ai["applicants"] > 0,
    (interaction_ai["bad_credit_rate"] - overall_bad_rate).round(2),
    np.nan
)

def flag_sample(n):
    if n == 0:
        return "Unobserved (N=0)"
    elif n < 10:
        return "Extremely Sparse (N<10)"
    elif n < 25:
        return "Sparse (N<25)"
    else:
        return "Stable"

interaction_ai["sample_flag"] = interaction_ai["applicants"].apply(flag_sample)

print("=== Credit Amount x Installment Rate Interaction — Detailed Table ===")
print(
    interaction_ai[
        [
            "credit_amount_quartile", "installment_rate",
            "applicants", "good_credit", "bad_credit",
            "bad_credit_rate", "difference_from_overall",
            "sample_flag"
        ]
    ].to_string(index=False)
)

print(f"\nTotal Applicants Accounted For: {interaction_ai['applicants'].sum()} (Target = 1,000)")
print(f"Total Defaults Accounted For:   {interaction_ai['bad_credit'].sum()} (Target = 300)")

In [165]:
pivot_ai_rate = interaction_ai.pivot_table(
    index="credit_amount_quartile",
    columns="installment_rate",
    values="bad_credit_rate",
    observed=False
)
pivot_ai_rate.columns = [f"Tier {c}" for c in pivot_ai_rate.columns]

print("=== Pivot Table: Bad-Credit Rate (%) by Credit Amount x Installment Rate ===")
print(pivot_ai_rate.round(2))

pivot_ai_count = interaction_ai.pivot_table(
    index="credit_amount_quartile",
    columns="installment_rate",
    values="applicants",
    observed=False
).fillna(0).astype(int)
pivot_ai_count.columns = [f"Tier {c}" for c in pivot_ai_count.columns]

print("\n=== Pivot Table: Applicant Counts by Credit Amount x Installment Rate ===")
print(pivot_ai_count)

In [166]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    pivot_ai_rate,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=60,
    mask=pivot_ai_count == 0,
    cbar_kws={"label": "Bad-Credit Rate (%)"},
    ax=ax
)

ax.set_title("Bad-Credit Rate (%) — Credit Amount x Installment Rate Interaction", fontsize=13, pad=14)
ax.set_xlabel("Installment Rate (Disposable Income Tier)", fontsize=11)
ax.set_ylabel("Credit Amount Quartile", fontsize=11)
ax.tick_params(axis="x", labelrotation=0)
ax.tick_params(axis="y", labelrotation=0)

plt.tight_layout()
plt.show()

In [167]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

observed_ai = interaction_ai[interaction_ai["applicants"] > 0]
stable_ai = interaction_ai[interaction_ai["applicants"] >= 25]
sparse_ai = interaction_ai[(interaction_ai["applicants"] < 25) & (interaction_ai["applicants"] >= 10)]
ext_sparse_ai = interaction_ai[(interaction_ai["applicants"] < 10) & (interaction_ai["applicants"] > 0)]
unobserved_ai = interaction_ai[interaction_ai["applicants"] == 0]

highest_raw = observed_ai.loc[observed_ai["bad_credit_rate"].idxmax()]
lowest_raw = observed_ai.loc[observed_ai["bad_credit_rate"].idxmin()]
highest_stable = stable_ai.loc[stable_ai["bad_credit_rate"].idxmax()]
lowest_stable = stable_ai.loc[stable_ai["bad_credit_rate"].idxmin()]
largest_cell = observed_ai.loc[observed_ai["applicants"].idxmax()]
smallest_obs = observed_ai.loc[observed_ai["applicants"].idxmin()]

print(f"Highest-risk Interaction (Empirical):        {highest_raw['credit_amount_quartile']} x Tier {highest_raw['installment_rate']} - {highest_raw['bad_credit_rate']:.2f}% ({int(highest_raw['applicants'])} app)")
print(f"Lowest-risk Interaction (Empirical):         {lowest_raw['credit_amount_quartile']} x Tier {lowest_raw['installment_rate']} - {lowest_raw['bad_credit_rate']:.2f}% ({int(lowest_raw['applicants'])} app)")
print(f"Highest-risk Stable Interaction (N >= 25):  {highest_stable['credit_amount_quartile']} x Tier {highest_stable['installment_rate']} - {highest_stable['bad_credit_rate']:.2f}% ({int(highest_stable['applicants'])} app)")
print(f"Lowest-risk Stable Interaction (N >= 25):   {lowest_stable['credit_amount_quartile']} x Tier {lowest_stable['installment_rate']} - {lowest_stable['bad_credit_rate']:.2f}% ({int(lowest_stable['applicants'])} app)")
print(f"Largest Combination:                         {largest_cell['credit_amount_quartile']} x Tier {largest_cell['installment_rate']} - {int(largest_cell['applicants'])} applicants")
print(f"Smallest Observed Combination:               {smallest_obs['credit_amount_quartile']} x Tier {smallest_obs['installment_rate']} - {int(smallest_obs['applicants'])} applicants")

print("\n--- Sample Size Breakdown ---")
print(f"Total possible combinations: 16 (4 amount quartiles x 4 installment tiers)")
print(f"Observed combinations:       {len(observed_ai)}")
print(f"Structurally unobserved:     {len(unobserved_ai)}")
print(f"Stable cells (N >= 25):      {len(stable_ai)}")
print(f"Sparse cells (10 <= N < 25): {len(sparse_ai)}")
print(f"Extremely sparse (N < 10):   {len(ext_sparse_ai)}")

print("\n--- Key Gradient & Compounding Observations ---")
print("1. Compounding Peak: Q4 Highest x Tier 4 reaches 49.28% default rate across 69 applicants (+19.28% above baseline).")
print("2. Installment Burden Gradient within Q3: Default rate increases monotonically: Tier 1 (11.11%) -> Tier 2 (19.40%) -> Tier 3 (23.68%) -> Tier 4 (29.00%).")
print("3. Balance Dominance in Q4: Even at Tier 1, Q4 applicants default at 41.67% (N=48), showing large principal exposure dominates low installment burden.")
print("4. Robust Matrix Density: 14 out of 16 cells have N >= 25, making this one of the most statistically stable interaction matrices in the dataset.")

## 38. Default Rate by Credit History and Loan Duration Interaction

In univariate analysis:
- **Credit History** (Section 14) revealed an apparent paradox where "critical accounts" exhibited lower empirical default rates (~17.0%), while "all paid back duly" showed higher default rates (~49.0%), heavily confounded by bank underwriting criteria.
- **Loan Duration** (Section 12) demonstrated a strong positive risk gradient: shorter terms (4-12 months) carried ~15.0% default rates, whereas long terms (37-48 months) spiked to ~52.4% default risk.

Here we examine the **5 x 5 two-way interaction (`credit_history` x `duration_bracket`)** to investigate:
1. How repayment track record interacts with credit horizon risk.
2. Whether short loan durations mitigate default risk even for critical credit history accounts.
3. How long loan horizons affect borrowers with existing or fully paid credit histories.

All findings represent **descriptive historical statistical associations**, not direct causal claims.

In [169]:
import numpy as np
import pandas as pd

# Define loan duration brackets
duration_bins = [0, 12, 24, 36, 48, 100]
duration_labels = ["4-12 months", "13-24 months", "25-36 months", "37-48 months", "49+ months"]
df["duration_bracket"] = pd.cut(
    df["duration_in_months"],
    bins=duration_bins,
    labels=duration_labels,
    right=True
)

history_names = {
    "A30": "No credits / All paid",
    "A31": "All paid back duly",
    "A32": "Existing paid duly",
    "A33": "Delay in past",
    "A34": "Critical account / Other existing"
}
df["credit_history_label"] = df["credit_history"].map(history_names)

history_order = [
    "Critical account / Other existing",
    "Delay in past",
    "Existing paid duly",
    "All paid back duly",
    "No credits / All paid"
]

overall_bad_rate = (df["credit_risk"] == 2).mean() * 100

interaction_hd = (
    df.groupby(["credit_history_label", "duration_bracket"], observed=False)
    .agg(
        applicants=("credit_risk", "size"),
        good_credit=("credit_risk", lambda x: (x == 1).sum()),
        bad_credit=("credit_risk", lambda x: (x == 2).sum()),
    )
    .reset_index()
)

interaction_hd["bad_credit_rate"] = np.where(
    interaction_hd["applicants"] > 0,
    (interaction_hd["bad_credit"] / interaction_hd["applicants"] * 100).round(2),
    np.nan
)
interaction_hd["difference_from_overall"] = np.where(
    interaction_hd["applicants"] > 0,
    (interaction_hd["bad_credit_rate"] - overall_bad_rate).round(2),
    np.nan
)

def flag_sample(n):
    if n == 0:
        return "Unobserved (N=0)"
    elif n < 10:
        return "Extremely Sparse (N<10)"
    elif n < 25:
        return "Sparse (N<25)"
    else:
        return "Stable"

interaction_hd["sample_flag"] = interaction_hd["applicants"].apply(flag_sample)

print("=== Credit History x Loan Duration Interaction - Detailed Table ===")
print(
    interaction_hd[
        [
            "credit_history_label", "duration_bracket",
            "applicants", "good_credit", "bad_credit",
            "bad_credit_rate", "difference_from_overall",
            "sample_flag"
        ]
    ].to_string(index=False)
)

print(f"\nTotal Applicants Accounted For: {interaction_hd['applicants'].sum()} (Target = 1,000)")
print(f"Total Defaults Accounted For:   {interaction_hd['bad_credit'].sum()} (Target = 300)")


In [170]:
pivot_hd_rate = interaction_hd.pivot_table(
    index="credit_history_label",
    columns="duration_bracket",
    values="bad_credit_rate",
    observed=False
).reindex(index=history_order, columns=duration_labels)

print("=== Pivot Table: Bad-Credit Rate (%) by Credit History x Duration ===")
print(pivot_hd_rate.round(2))

pivot_hd_count = interaction_hd.pivot_table(
    index="credit_history_label",
    columns="duration_bracket",
    values="applicants",
    observed=False
).reindex(index=history_order, columns=duration_labels).fillna(0).astype(int)

print("\n=== Pivot Table: Applicant Counts by Credit History x Duration ===")
print(pivot_hd_count)


In [171]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(11, 6))

sns.heatmap(
    pivot_hd_rate,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=75,
    mask=pivot_hd_count == 0,
    cbar_kws={"label": "Bad-Credit Rate (%)"},
    ax=ax
)

ax.set_title("Bad-Credit Rate (%) - Credit History x Loan Duration Interaction", fontsize=13, pad=14)
ax.set_xlabel("Loan Duration Bracket", fontsize=11)
ax.set_ylabel("Credit History", fontsize=11)
ax.tick_params(axis="x", labelrotation=0)
ax.tick_params(axis="y", labelrotation=0)

plt.tight_layout()
plt.show()


In [172]:
print(f"Overall Portfolio Default Rate: {overall_bad_rate:.2f}%\n")

observed_hd = interaction_hd[interaction_hd["applicants"] > 0]
stable_hd = interaction_hd[interaction_hd["applicants"] >= 25]
sparse_hd = interaction_hd[(interaction_hd["applicants"] < 25) & (interaction_hd["applicants"] >= 10)]
ext_sparse_hd = interaction_hd[(interaction_hd["applicants"] < 10) & (interaction_hd["applicants"] > 0)]
unobserved_hd = interaction_hd[interaction_hd["applicants"] == 0]

highest_raw = observed_hd.loc[observed_hd["bad_credit_rate"].idxmax()]
lowest_raw = observed_hd.loc[observed_hd["bad_credit_rate"].idxmin()]
highest_stable = stable_hd.loc[stable_hd["bad_credit_rate"].idxmax()]
lowest_stable = stable_hd.loc[stable_hd["bad_credit_rate"].idxmin()]
largest_cell = observed_hd.loc[observed_hd["applicants"].idxmax()]

print(f"Highest-risk Interaction (Empirical):        {highest_raw['credit_history_label']} x {highest_raw['duration_bracket']} - {highest_raw['bad_credit_rate']:.2f}% ({int(highest_raw['applicants'])} app)")
print(f"Lowest-risk Interaction (Empirical):         {lowest_raw['credit_history_label']} x {lowest_raw['duration_bracket']} - {lowest_raw['bad_credit_rate']:.2f}% ({int(lowest_raw['applicants'])} app)")
print(f"Highest-risk Stable Interaction (N >= 25):  {highest_stable['credit_history_label']} x {highest_stable['duration_bracket']} - {highest_stable['bad_credit_rate']:.2f}% ({int(highest_stable['applicants'])} app)")
print(f"Lowest-risk Stable Interaction (N >= 25):   {lowest_stable['credit_history_label']} x {lowest_stable['duration_bracket']} - {lowest_stable['bad_credit_rate']:.2f}% ({int(lowest_stable['applicants'])} app)")
print(f"Largest Combination:                         {largest_cell['credit_history_label']} x {largest_cell['duration_bracket']} - {int(largest_cell['applicants'])} applicants")

print("\n--- Sample Size Breakdown ---")
print(f"Total possible combinations: 25 (5 credit history categories x 5 duration brackets)")
print(f"Observed combinations:       {len(observed_hd)}")
print(f"Structurally unobserved:     {len(unobserved_hd)}")
print(f"Stable cells (N >= 25):      {len(stable_hd)}")
print(f"Sparse cells (10 <= N < 25): {len(sparse_hd)}")
print(f"Extremely sparse (N < 10):   {len(ext_sparse_hd)}")

print("\n--- Key Interaction Observations ---")
print("1. Duration Escalation in 'Existing Paid Duly': Risk increases steeply with duration: 4-12m (24.15%, N=207) -> 13-24m (30.14%, N=219) -> 25-36m (43.94%, N=66) -> 37-48m (63.33%, N=30).")
print("2. Protective Effect of Short Duration on Critical Accounts: 'Critical Account' applicants with short terms (4-12m) exhibit an exceptionally low default rate of 8.70% (N=115).")
print("3. Underwriting Policy Confounding: 'All Paid Back Duly' applicants exhibit high default rates even in short durations (60.0% in 4-12m, N=15), confirming substantial selection bias.")
print("4. Modeling Implication: Capturing non-linear interactions between duration and credit history is essential for accurate risk scoring.")
